In [32]:
!pip install numpy==1.26.4



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



## Step 0 – Environment Setup

This step prepares the Python environment by importing all required
libraries for data manipulation and visualizatio


In [33]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Step 1 – Data Loading

#The datasets are loaded from locally uploaded CSV files within the Jupyter environment.
#These files represent structured extracts of the original SQL database.

#Primary keys and foreign keys were enforced at the database level prior to extraction.
#Therefore, the datasets are expected to be relationally consistent and ready for analysis.

In [34]:
# ============================================================
# Load all CSV tables into Pandas DataFrames
# Files are assumed to be in the same directory as the notebook
# ============================================================

import pandas as pd

# Core tables
core_customers = pd.read_csv("core_customers.csv")
core_orders = pd.read_csv("core_orders.csv")
core_order_items = pd.read_csv("core_order_items.csv")
core_payments = pd.read_csv("core_payments.csv")
core_products = pd.read_csv("core_products.csv")
core_reviews = pd.read_csv("core_reviews.csv")
core_sellers = pd.read_csv("core_sellers.csv")

# Reference tables
ref_category_translation = pd.read_csv("ref_category_translation.csv")  
ref_geolocation = pd.read_csv("ref_geolocation.csv")

print("✅ All tables loaded successfully")

✅ All tables loaded successfully


In [35]:

tables_map = {
    "core_customers": core_customers,
    "core_order_items": core_order_items,
    "core_orders": core_orders,
    "core_payments": core_payments,
    "core_products": core_products,
    "core_reviews": core_reviews,
    "core_sellers": core_sellers,
    "ref_category_translation": ref_category_translation,
    "ref_geolocation": ref_geolocation
}

dtype_report = []

for table_name, df in tables_map.items():
    total_rows = len(df)
    
    for col in df.columns:
        # Count normal missing values: NaN, None, NaT
        missing_count = df[col].isna().sum()
        
        # Count empty strings only for object/string columns
        if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
            empty_string_count = df[col].astype(str).str.strip().eq("").sum()
        else:
            empty_string_count = 0
        
        # Total null-like values = missing + empty strings
        total_null_count = missing_count + empty_string_count
        
        # Percentages
        missing_percentage = (missing_count / total_rows) * 100 if total_rows > 0 else 0
        empty_string_percentage = (empty_string_count / total_rows) * 100 if total_rows > 0 else 0
        total_null_percentage = (total_null_count / total_rows) * 100 if total_rows > 0 else 0
        
        # Count duplicated values in the column
        duplicate_count = df[col].duplicated().sum()
        duplicate_percentage = (duplicate_count / total_rows) * 100 if total_rows > 0 else 0
        
        dtype_report.append({
            "Table": table_name,
            "Column": col,
            "Data Type": df[col].dtype,
            "Total Rows": total_rows,
            "Missing Count": missing_count,
            "Empty String Count": empty_string_count,
            "Total Null-like Count": total_null_count,
            "Missing %": round(missing_percentage, 2),
            "Empty String %": round(empty_string_percentage, 2),
            "Total Null-like %": round(total_null_percentage, 2),
            "Duplicate Count": duplicate_count,
            "Duplicate %": round(duplicate_percentage, 2)
        })

dtype_report_df = pd.DataFrame(dtype_report)

dtype_report_df

,Table,Column,Data Type,Total Rows,Missing Count,Empty String Count,Total Null-like Count,Missing %,Empty String %,Total Null-like %,Duplicate Count,Duplicate %
0,core_customers,customer_id,str,99441,0,0,0,0.00,0.00,0.00,0,0.00
1,core_customers,customer_unique_id,str,99441,0,0,0,0.00,0.00,0.00,3345,3.36
2,core_customers,customer_zip_code_prefix,int64,99441,0,0,0,0.00,0.00,0.00,84447,84.92
3,core_customers,customer_city,str,99441,0,0,0,0.00,0.00,0.00,95322,95.86
4,core_customers,customer_state,str,99441,0,0,0,0.00,0.00,0.00,99414,99.97
5,core_order_items,order_id,str,112650,0,0,0,0.00,0.00,0.00,13984,12.41
6,core_order_items,order_item_id,int64,112650,0,0,0,0.00,0.00,0.00,112629,99.98
7,core_order_items,product_id,str,112650,0,0,0,0.00,0.00,0.00,79699,70.75
8,core_order_items,seller_id,str,112650,0,0,0,0.00,0.00,0.00,109555,97.25
9,core_order_items,shipping_limit_date,float64,112650,0,0,0,0.00,0.00,0.00,19332,17.16


# Problem No. 1
Identifying date columns types issue and dignosis missing content in these columns with corrective actions

In [36]:

# Define the date columns that need to be converted in each DataFrame
date_columns_map = {
    "core_orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    
    "core_order_items": [
        "shipping_limit_date"
    ],
    
    "core_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Map DataFrame names to the actual DataFrames
dataframes_map = {
    "core_orders": core_orders,
    "core_order_items": core_order_items,
    "core_reviews": core_reviews
}

# Convert the selected columns to datetime
for df_name, date_cols in date_columns_map.items():
    df = dataframes_map[df_name]
    
    print(f"\n===== {df_name} =====")
    
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )
            
            print(
                f"{col} --> {df[col].dtype} | "
                f"Missing/Invalid dates: {df[col].isna().sum()}"
            )
        else:
            print(f"{col} --> Column not found")


===== core_orders =====
order_purchase_timestamp --> datetime64[us] | Missing/Invalid dates: 0
order_approved_at --> datetime64[us] | Missing/Invalid dates: 160
order_delivered_carrier_date --> datetime64[us] | Missing/Invalid dates: 1783
order_delivered_customer_date --> datetime64[us] | Missing/Invalid dates: 2965
order_estimated_delivery_date --> datetime64[us] | Missing/Invalid dates: 0

===== core_order_items =====
shipping_limit_date --> datetime64[ns] | Missing/Invalid dates: 0

===== core_reviews =====
review_creation_date --> datetime64[us] | Missing/Invalid dates: 0
review_answer_timestamp --> datetime64[us] | Missing/Invalid dates: 0


In [37]:
# identify invalid and missing data
import pandas as pd

def convert_and_audit_dates(df, cols, fmt=None):
    for col in cols:
        if col not in df.columns:
            print(f"{col} --> Column not found")
            continue

        s = df[col]

        # Missing before conversion: NaN / None / empty strings
        missing_before = s.isna() | s.astype(str).str.strip().eq("")

        # Convert
        dt = pd.to_datetime(s, errors="coerce", format=fmt)

        # Invalid means: not missing_before but became NaT after conversion
        invalid = (~missing_before) & dt.isna()

        df[col] = dt.astype("datetime64[ns]")  # توحيد النوع ns بدل us

        print(
            f"{col} --> {df[col].dtype} | "
            f"Missing: {missing_before.sum()} | Invalid: {invalid.sum()}"
        )

        if invalid.sum() > 0:
            examples = s[invalid].astype(str).head(5).tolist()
            print("   Examples of invalid values:", examples)


date_columns_map = {
    "core_orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "core_order_items": ["shipping_limit_date"],
    "core_reviews": ["review_creation_date", "review_answer_timestamp"]
}

dataframes_map = {
    "core_orders": core_orders,
    "core_order_items": core_order_items,
    "core_reviews": core_reviews
}

for df_name, cols in date_columns_map.items():
    print(f"\n===== {df_name} =====")
    convert_and_audit_dates(dataframes_map[df_name], cols)



===== core_orders =====
order_purchase_timestamp --> datetime64[ns] | Missing: 0 | Invalid: 0
order_approved_at --> datetime64[ns] | Missing: 160 | Invalid: 0
order_delivered_carrier_date --> datetime64[ns] | Missing: 1783 | Invalid: 0
order_delivered_customer_date --> datetime64[ns] | Missing: 2965 | Invalid: 0
order_estimated_delivery_date --> datetime64[ns] | Missing: 0 | Invalid: 0

===== core_order_items =====
shipping_limit_date --> datetime64[ns] | Missing: 0 | Invalid: 0

===== core_reviews =====
review_creation_date --> datetime64[ns] | Missing: 0 | Invalid: 0
review_answer_timestamp --> datetime64[ns] | Missing: 0 | Invalid: 0


In [38]:
# Columns with missing dates in core_orders
date_problem_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

# Check missing values by order_status
for col in date_problem_cols:
    print(f"\n===== Missing analysis for: {col} =====")
    
    result = (
        core_orders
        .assign(is_missing=core_orders[col].isna())
        .groupby("order_status")["is_missing"]
        .agg(
            total_orders="count",
            missing_count="sum"
        )
        .reset_index()
    )
    
    result["missing_percentage"] = (
        result["missing_count"] / result["total_orders"] * 100
    ).round(2)
    
    print(result)


===== Missing analysis for: order_approved_at =====
  order_status  total_orders  missing_count  missing_percentage
0     approved             2              0                0.00
1     canceled           625            141               22.56
2      created             5              5              100.00
3    delivered         96478             14                0.01
4     invoiced           314              0                0.00
5   processing           301              0                0.00
6      shipped          1107              0                0.00
7  unavailable           609              0                0.00

===== Missing analysis for: order_delivered_carrier_date =====
  order_status  total_orders  missing_count  missing_percentage
0     approved             2              2               100.0
1     canceled           625            550                88.0
2      created             5              5               100.0
3    delivered         96478              2        

## Key observations from the missingness report

order_approved_at is 100% missing for created and ~22.6% missing for canceled — this is expected because many orders are canceled before approval, and “created” orders often never reach approval.

order_delivered_carrier_date is 0% missing for shipped and ~0% missing for delivered — expected because “shipped” implies the package was handed to the carrier.

order_delivered_customer_date is 100% missing for shipped — expected because shipped ≠ delivered.

The only “data quality concern” is a very small number of orders marked as delivered but missing one of the delivery-related timestamps (e.g., delivered but missing delivered_customer_date). These are rare inconsistencies that should be handled explicitly (flagged and excluded from certain KPIs), not silently “fixed”.

Why this is not a parsing/format issue
Your latest audit shows Invalid = 0 for all date columns — so the timestamps are valid when present. The missing values represent events that did not occur (or were not recorded) rather than “broken date strings”.

## ✅ Recommendation (what we will do)
Recommended approach (clean + honest + analysis-friendly)

Keep the original date columns unchanged (do not overwrite the truth).
Create data-quality flags to mark inconsistent records (e.g., delivered but missing a delivered date).
For KPI calculations:

Use a clean subset (e.g., only delivered orders with the necessary timestamps present).


If you must create a complete timestamp for modeling/EDA:

Create a separate filled column (e.g., delivered date = actual if available else estimated) and add a boolean flag indicating imputation.



This gives you:

Accurate business interpretation (missing ≠ error)
Reproducible KPIs
Safe features for modeling without corrupting raw data

In [39]:
# run datetime conversion (standardized to datetime64)
import pandas as pd

# Convert relevant columns to datetime (keeps NaT for missing values)
date_columns_map = {
    "core_orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "core_order_items": ["shipping_limit_date"],
    "core_reviews": ["review_creation_date", "review_answer_timestamp"]
}

dataframes_map = {
    "core_orders": core_orders,
    "core_order_items": core_order_items,
    "core_reviews": core_reviews
}

for df_name, cols in date_columns_map.items():
    df = dataframes_map[df_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").astype("datetime64[ns]")

print("✅ Datetime conversion completed (datetime64[ns])")


✅ Datetime conversion completed (datetime64[ns])


In [40]:
# Missingness report by order_status (Reusable function)
import pandas as pd

def missing_by_status(df, date_col, status_col="order_status"):
    """
    Build a missingness report for a given column grouped by order status.

    Why this matters:
    - Missing dates may be perfectly normal for some statuses (e.g., 'created', 'canceled').
    - Grouping by status helps you distinguish expected missingness from true data quality issues.

    Output columns:
    - total_orders: number of rows per status
    - missing_count: number of missing (NaT/NaN) values in the target column
    - missing_percentage: missing_count / total_orders * 100
    """

    # Group by status and compute total rows and missing values for the selected column
    report = (
        df.groupby(status_col)[date_col]
        .agg(
            total_orders="size",                         # Total rows in each status
            missing_count=lambda s: s.isna().sum()        # Count NaN/NaT values
        )
        .reset_index()
    )

    # Calculate missing percentage per status
    report["missing_percentage"] = (report["missing_count"] / report["total_orders"] * 100).round(2)

    # Sort: show the highest missing% statuses first (then by volume)
    return report.sort_values(["missing_percentage", "total_orders"], ascending=[False, False])


# Generate reports for key timestamp columns in core_orders
for col in [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]:
    print(f"\n===== Missing analysis for: {col} =====")
    display(missing_by_status(core_orders, col))


===== Missing analysis for: order_approved_at =====


,order_status,total_orders,missing_count,missing_percentage
2,created,5,5,100.00
1,canceled,625,141,22.56
3,delivered,96478,14,0.01
6,shipped,1107,0,0.00
7,unavailable,609,0,0.00
4,invoiced,314,0,0.00
5,processing,301,0,0.00
0,approved,2,0,0.00



===== Missing analysis for: order_delivered_carrier_date =====


,order_status,total_orders,missing_count,missing_percentage
7,unavailable,609,609,100.0
4,invoiced,314,314,100.0
5,processing,301,301,100.0
2,created,5,5,100.0
0,approved,2,2,100.0
1,canceled,625,550,88.0
3,delivered,96478,2,0.0
6,shipped,1107,0,0.0



===== Missing analysis for: order_delivered_customer_date =====


,order_status,total_orders,missing_count,missing_percentage
6,shipped,1107,1107,100.00
7,unavailable,609,609,100.00
4,invoiced,314,314,100.00
5,processing,301,301,100.00
2,created,5,5,100.00
0,approved,2,2,100.00
1,canceled,625,619,99.04
3,delivered,96478,8,0.01


## Step 1 — Interpretation of Missing Date Results

### 1) `order_approved_at`

#### Observations
- `created` orders have **100% missing values** in `order_approved_at`.  
  This is expected because these orders were only created and have not reached the approval stage yet.

- `canceled` orders have **141 missing values out of 625 orders (22.56%)**.  
  This is logical because some orders were canceled before they were approved.

- `delivered` orders have **14 missing values out of 96,478 orders (0.01%)**.  
  This is a data quality anomaly because a delivered order is generally expected to have an approval timestamp.

#### Recommendation
Create a data quality flag for delivered orders with missing `order_approved_at`, instead of filling the value or deleting the records immediately.

---

### 2) `order_delivered_carrier_date`

#### Observations
- `approved`, `created`, `processing`, `invoiced`, and `unavailable` orders have mostly or fully missing values.  
  This is expected because these orders have not reached the shipping/carrier stage.

- `canceled` orders have **550 missing values out of 625 orders (88.00%)**.  
  This is logical because most canceled orders were not handed over to the carrier.

- `shipped` orders have **0 missing values**.  
  This is consistent with the business meaning of `shipped`, as these orders should already have been handed over to the carrier.

- `delivered` orders have only **2 missing values**.  
  This is a data quality anomaly because an order delivered to the customer is expected to have passed through the carrier stage.

#### Recommendation
Create a data quality flag for delivered orders with missing `order_delivered_carrier_date`.

---

### 3) `order_delivered_customer_date`

#### Observations
- `shipped` orders have **100% missing values** in `order_delivered_customer_date`.  
  This is expected because `shipped` means the order has been shipped but has not necessarily been delivered to the customer yet.

- `created`, `approved`, `processing`, `invoiced`, and `unavailable` orders have **100% missing values**.  
  This is also expected because these orders have not reached the final customer delivery stage.

- `canceled` orders have **619 missing values out of 625 orders (99.04%)**.  
  This is expected because canceled orders are usually not delivered to the customer.

- `delivered` orders have only **8 missing values**.  
  This is the most important anomaly because an order marked as `delivered` should normally have a customer delivery timestamp.

#### Recommendation
Create a data quality flag for the 8 delivered records with missing `order_delivered_customer_date`.

For delivery-related KPIs, these records should be excluded because the actual delivery date is missing.

If the data is needed for modeling or exploratory analysis, a separate filled/imputed column can be created using `order_estimated_delivery_date`, but the original column should remain unchanged and an imputation flag should be added.

## Decision After Missing Date Analysis

Based on the missingness analysis by `order_status`, most missing values in the order date columns are expected and consistent with the business lifecycle of an order.

For example:
- Orders with statuses such as `created`, `approved`, `processing`, `invoiced`, `shipped`, `unavailable`, and `canceled` may naturally have missing delivery-related timestamps.
- The only records that require special attention are orders marked as `delivered` but missing one or more critical timestamps.

### Data Quality Decision

At this stage, we will not calculate delivery KPIs yet.

Instead, we will:
1. Keep the original date columns unchanged.
2. Create data quality flags for inconsistent delivered orders.
3. Use these flags later during the EDA phase to decide whether to exclude or separately analyze these records.

### KPI Decision

Delivery-related KPIs such as:
- delivery time in days,
- late delivery flag,
- on-time delivery rate,
- carrier-to-customer delivery duration,

will be created later in the EDA section, after the data quality flags have been added and reviewed.

In [41]:
# Create data quality flags for delivered orders with missing critical timestamps.
# These flags help identify rare inconsistencies without changing the original data.

core_orders["dq_delivered_missing_approved_at"] = (
    (core_orders["order_status"] == "delivered") &
    (core_orders["order_approved_at"].isna())
)

core_orders["dq_delivered_missing_carrier_date"] = (
    (core_orders["order_status"] == "delivered") &
    (core_orders["order_delivered_carrier_date"].isna())
)

core_orders["dq_delivered_missing_customer_date"] = (
    (core_orders["order_status"] == "delivered") &
    (core_orders["order_delivered_customer_date"].isna())
)

# Create one combined flag for any delivered order with at least one critical missing timestamp.
core_orders["dq_delivered_any_missing_critical_date"] = (
    core_orders["dq_delivered_missing_approved_at"] |
    core_orders["dq_delivered_missing_carrier_date"] |
    core_orders["dq_delivered_missing_customer_date"]
)

# Summarize the number of records flagged by each data quality rule.
dq_date_flags_summary = core_orders[
    [
        "dq_delivered_missing_approved_at",
        "dq_delivered_missing_carrier_date",
        "dq_delivered_missing_customer_date",
        "dq_delivered_any_missing_critical_date"
    ]
].sum().reset_index()

dq_date_flags_summary.columns = ["data_quality_flag", "flagged_records"]

print("✅ Data quality flags created successfully")
display(dq_date_flags_summary)

✅ Data quality flags created successfully


,data_quality_flag,flagged_records
0,dq_delivered_missing_approved_at,14
1,dq_delivered_missing_carrier_date,2
2,dq_delivered_missing_customer_date,8
3,dq_delivered_any_missing_critical_date,23


# Problem No. 2 
Dealing with missing values in columns other than date columns

## Review Comments Missingness Analysis

The profiling report shows a high percentage of missing values in review text fields:

- `review_comment_title` has around 88.34% null-like values.
- `review_comment_message` has around 58.73% null-like values.

### Interpretation

This missingness is expected in customer review data because customers are usually allowed to submit a review score without writing a comment title or comment message.

Therefore, missing text comments should not be treated as invalid records.

### Data Quality Decision

At this stage:
- We will not delete reviews with missing comment text.
- We will not fill missing comments with artificial text.
- We will create indicator flags to show whether a review has a title and/or a message.
- These flags can be used later in EDA to compare review scores between reviews with and without written comments.

### Recommended Treatment

Create the following flags:
- `has_review_comment_title`
- `has_review_comment_message`
- `has_any_review_text`

These flags preserve the original data while making the missingness useful for analysis.

In [42]:
# Diagnose missingness in review text fields.
# This checks missing values, empty strings, and whitespace-only strings.

review_text_cols = ["review_comment_title", "review_comment_message"]

review_text_diagnosis = []

for col in review_text_cols:
    # Count true missing values: NaN / None
    missing_count = core_reviews[col].isna().sum()
    
    # Count empty strings after converting to string and stripping spaces
    empty_string_count = (
        core_reviews[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    # Count non-empty text values
    non_empty_text_count = (
        ~core_reviews[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    ).sum()
    
    review_text_diagnosis.append({
        "column": col,
        "total_rows": len(core_reviews),
        "missing_count": missing_count,
        "empty_or_blank_count": empty_string_count,
        "non_empty_text_count": non_empty_text_count,
        "empty_or_blank_percentage": round(empty_string_count / len(core_reviews) * 100, 2),
        "non_empty_text_percentage": round(non_empty_text_count / len(core_reviews) * 100, 2)
    })

review_text_diagnosis_df = pd.DataFrame(review_text_diagnosis)

print("✅ Review text missingness diagnosis completed")
display(review_text_diagnosis_df)

✅ Review text missingness diagnosis completed


,column,total_rows,missing_count,empty_or_blank_count,non_empty_text_count,empty_or_blank_percentage,non_empty_text_percentage
0,review_comment_title,99222,87655,87657,11565,88.34,11.66
1,review_comment_message,99222,58247,58274,40948,58.73,41.27


In [43]:
# Create flags indicating whether review text fields are available.
# These flags preserve the original columns and make missingness useful for analysis.

core_reviews["has_review_comment_title"] = (
    ~core_reviews["review_comment_title"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

core_reviews["has_review_comment_message"] = (
    ~core_reviews["review_comment_message"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

# Combined flag: True if the review has either a title or a message
core_reviews["has_any_review_text"] = (
    core_reviews["has_review_comment_title"] |
    core_reviews["has_review_comment_message"]
)

# Summarize the new flags
review_text_flags_summary = core_reviews[
    [
        "has_review_comment_title",
        "has_review_comment_message",
        "has_any_review_text"
    ]
].sum().reset_index()

review_text_flags_summary.columns = ["review_text_flag", "true_count"]

review_text_flags_summary["true_percentage"] = (
    review_text_flags_summary["true_count"] / len(core_reviews) * 100
).round(2)

print("✅ Review text availability flags created")
display(review_text_flags_summary)

✅ Review text availability flags created


,review_text_flag,true_count,true_percentage
0,has_review_comment_title,11565,11.66
1,has_review_comment_message,40948,41.27
2,has_any_review_text,42685,43.02


In [44]:
# Compare review scores based on whether the review has written text.
# This helps understand whether missing comments are random or related to customer satisfaction.

review_score_by_text_flag = (
    core_reviews
    .groupby("has_any_review_text")["review_score"]
    .agg(
        review_count="size",
        average_score="mean",
        median_score="median",
        min_score="min",
        max_score="max"
    )
    .reset_index()
)

review_score_by_text_flag["average_score"] = review_score_by_text_flag["average_score"].round(2)

print("✅ Review score comparison by text availability")
display(review_score_by_text_flag)

✅ Review score comparison by text availability


,has_any_review_text,review_count,average_score,median_score,min_score,max_score
0,False,56537,4.38,5.0,1,5
1,True,42685,3.70,5.0,1,5


## Review Text Availability Insight

The comparison shows that reviews without written text have a higher average score than reviews with written text.

- Reviews without text: **56,537 reviews**, average score = **4.38**
- Reviews with text: **42,685 reviews**, average score = **3.70**

### Interpretation

Customers who do not write comments usually give higher scores, while customers who write comments tend to have a lower average score. This suggests that written comments may often be used when customers want to explain an issue or provide more detailed feedback.

### Data Treatment Decision

Missing review comments should not be treated as invalid data.

We will:
- Keep all review records.
- Not delete reviews with missing comments.
- Not fill missing comments with artificial text.
- Use the review text availability flags in EDA:
  - `has_review_comment_title`
  - `has_review_comment_message`
  - `has_any_review_text`

These flags will help analyze whether written feedback is associated with lower review scores.

## Review Comments Treatment Summary

The high missingness in `review_comment_title` and `review_comment_message` is expected because customers can submit review scores without writing textual comments.

### Treatment Applied

- Original review text columns were kept unchanged.
- Text availability flags were created:
  - `has_review_comment_title`
  - `has_review_comment_message`
  - `has_any_review_text`

### Decision

Reviews with missing comments will not be removed because they still contain valid review scores.

The new text availability flags will be used later in the EDA phase to analyze whether written comments are associated with different review scores.

# ✅ Problem 3 — Missing value in ref_category_translation.product_category_name_english

## Category Translation Missing Value

The `ref_category_translation` table has one missing value in `product_category_name_english`.

This missing value belongs to the category:

- `product_category_name = unknown`

### Interpretation

This is not a major data quality issue.  
The original category is already labeled as `unknown`, so the missing English translation can be safely filled with `"unknown"`.

### Treatment Decision

- Keep the row.
- Do not delete the category.
- Fill the missing English translation with `"unknown"`.

This keeps the translation table complete and avoids missing category names after joining it with the products table.

In [45]:
# Identify rows where the English category translation is missing.
# This helps confirm which category needs treatment.

missing_category_translation = ref_category_translation[
    ref_category_translation["product_category_name_english"].isna()
].copy()

print("Number of missing English category translations:", len(missing_category_translation))

display(missing_category_translation)

Number of missing English category translations: 1


,product_category_name,product_category_name_english
72,unknown,NaN


In [46]:
# Fill missing English category names with "unknown".
# This is appropriate because the original category name is also "unknown".

ref_category_translation["product_category_name_english"] = (
    ref_category_translation["product_category_name_english"]
    .fillna("unknown")
)

print("✅ Missing English category translations filled with 'unknown'")
print("Remaining missing values:",
      ref_category_translation["product_category_name_english"].isna().sum())

✅ Missing English category translations filled with 'unknown'
Remaining missing values: 0


In [47]:
# Re-check after filling missing/blank values.

translation_quality_check_after = []

for col in ref_category_translation.columns:
    missing_count = ref_category_translation[col].isna().sum()
    
    empty_string_count = (
        ref_category_translation[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    translation_quality_check_after.append({
        "column": col,
        "missing_count": missing_count,
        "empty_or_blank_count": empty_string_count,
        "total_null_like_count": missing_count + empty_string_count
    })

translation_quality_check_after_df = pd.DataFrame(translation_quality_check_after)

display(translation_quality_check_after_df)

,column,missing_count,empty_or_blank_count,total_null_like_count
0,product_category_name,0,0,0
1,product_category_name_english,0,0,0


## Category Translation Treatment Summary

The missing English category translation was found in the `unknown` category.

### Treatment Applied

The missing value in `product_category_name_english` was filled with `"unknown"`.

### Decision

This treatment is safe because the original category name is already `unknown`.  
The row was kept to preserve completeness and avoid missing values after future joins with the products table.


In [48]:
# Diagnose whether the category translation issue is related to missing or unmatched categories in core_products.

# 1) Check missing or blank product categories in core_products
missing_product_category_count = (
    core_products["product_category_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Missing or blank product_category_name in core_products:", missing_product_category_count)


# 2) Check categories that exist in core_products but do not exist in ref_category_translation
product_categories = set(
    core_products["product_category_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

translation_categories = set(
    ref_category_translation["product_category_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

unmatched_categories = sorted(product_categories - translation_categories)

print("Number of product categories not found in translation table:", len(unmatched_categories))
print("Unmatched categories:", unmatched_categories)


# 3) Perform a left join to check if any products still have missing English category after translation
products_translation_check = core_products.merge(
    ref_category_translation,
    on="product_category_name",
    how="left"
)

missing_english_after_join = (
    products_translation_check["product_category_name_english"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Products without English category after join:", missing_english_after_join)

Missing or blank product_category_name in core_products: 0
Number of product categories not found in translation table: 0
Unmatched categories: []
Products without English category after join: 0


## Category Translation Relationship Check

After fixing the missing value in `ref_category_translation.product_category_name_english`, a relationship check was performed between `core_products` and `ref_category_translation`.

### Results

- Missing or blank `product_category_name` in `core_products`: 0
- Product categories not found in the translation table: 0
- Unmatched categories: none
- Products without English category after joining with the translation table: 0

### Interpretation

The missing translation issue was isolated to the translation reference table and did not indicate a related missing category issue in `core_products`.

All product categories in `core_products` have valid matches in `ref_category_translation`, and all products receive an English category after the join.

### Decision

The category translation issue is considered fully resolved.  
No additional treatment is required before moving to duplicate and key uniqueness checks.

# Problem No. 4 : Duplicates

In [49]:


# Step 1: Check full-row duplicates in each table

tables_map = {
    "core_customers": core_customers,
    "core_order_items": core_order_items,
    "core_orders": core_orders,
    "core_payments": core_payments,
    "core_products": core_products,
    "core_reviews": core_reviews,
    "core_sellers": core_sellers,
    "ref_category_translation": ref_category_translation,
    "ref_geolocation": ref_geolocation
}

full_duplicate_report = []

for table_name, df in tables_map.items():
    total_rows = len(df)
    duplicate_rows = df.duplicated().sum()
    duplicate_percentage = round((duplicate_rows / total_rows) * 100, 2) if total_rows > 0 else 0
    
    full_duplicate_report.append({
        "Table": table_name,
        "Total Rows": total_rows,
        "Full Duplicate Rows": duplicate_rows,
        "Duplicate %": duplicate_percentage
    })

full_duplicate_report_df = pd.DataFrame(full_duplicate_report)

full_duplicate_report_df


,Table,Total Rows,Full Duplicate Rows,Duplicate %
0,core_customers,99441,0,0.00
1,core_order_items,112650,0,0.00
2,core_orders,99441,0,0.00
3,core_payments,103886,0,0.00
4,core_products,32951,0,0.00
5,core_reviews,99222,0,0.00
6,core_sellers,3095,0,0.00
7,ref_category_translation,74,0,0.00
8,ref_geolocation,1000163,261831,26.18


## Full Duplicate Rows Analysis

A full row duplicate check was performed across all datasets.

### Results

All tables have 0 full duplicate rows except `ref_geolocation`.

`ref_geolocation` contains:

- Total rows: 1,000,163
- Full duplicate rows: 261,831
- Duplicate percentage: 26.18%

### Interpretation

The duplicated rows in `ref_geolocation` are exact duplicate records, meaning that the same ZIP code, latitude, longitude, city, and state combination appears more than once.

Since these rows are fully identical, they do not add new information and may increase memory usage or create unnecessary duplication during joins.

### Treatment Decision

- Keep the original `ref_geolocation` unchanged.
- Create a cleaned version named `ref_geolocation_clean`.
- Remove only exact full duplicate rows using `drop_duplicates()`.
- Verify that no full duplicate rows remain after cleaning.

In [50]:
# Inspect exact full duplicate rows in ref_geolocation.
# keep=False returns all rows that are part of a duplicate group.

geo_full_duplicates = ref_geolocation[
    ref_geolocation.duplicated(keep=False)
].copy()

print("Total full duplicate rows in ref_geolocation:", len(geo_full_duplicates))

display(
    geo_full_duplicates
    .sort_values(list(ref_geolocation.columns))
    .head(30)
)

Total full duplicate rows in ref_geolocation: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


In [51]:
# Create a cleaned version of ref_geolocation by removing exact full duplicate rows.
# The original ref_geolocation table remains unchanged.

ref_geolocation_clean = ref_geolocation.drop_duplicates().copy()

print("Original ref_geolocation rows:", len(ref_geolocation))
print("Cleaned ref_geolocation rows:", len(ref_geolocation_clean))
print("Removed duplicate rows:", len(ref_geolocation) - len(ref_geolocation_clean))
print(
    "Removed duplicate percentage:",
    round((len(ref_geolocation) - len(ref_geolocation_clean)) / len(ref_geolocation) * 100, 2),
    "%"
)

Original ref_geolocation rows: 1000163
Cleaned ref_geolocation rows: 738332
Removed duplicate rows: 261831
Removed duplicate percentage: 26.18 %


In [52]:
# Verify that no exact full duplicate rows remain in the cleaned geolocation table.

remaining_full_duplicates = ref_geolocation_clean.duplicated().sum()

print("Remaining full duplicate rows in ref_geolocation_clean:", remaining_full_duplicates)

if remaining_full_duplicates == 0:
    print("✅ Full duplicate rows successfully removed from ref_geolocation_clean.")
else:
    print("⚠️ There are still duplicate rows that need investigation.")


Remaining full duplicate rows in ref_geolocation_clean: 0
✅ Full duplicate rows successfully removed from ref_geolocation_clean.


## Full Duplicate Rows Treatment Summary — `ref_geolocation`

A full duplicate row check showed that `ref_geolocation` was the only table containing exact duplicate rows.

### Results Before Cleaning

- Original rows: **1,000,163**
- Full duplicate rows removed: **261,831**
- Duplicate percentage removed: **26.18%**

### Treatment Applied

A cleaned version of the geolocation table was created:

`ref_geolocation_clean`

Exact duplicate rows were removed using `drop_duplicates()`, while keeping the original `ref_geolocation` table unchanged.

### Results After Cleaning

- Cleaned rows: **738,332**
- Remaining full duplicate rows in `ref_geolocation_clean`: **0**

### Interpretation

The removed rows were exact duplicates, meaning the same geolocation record appeared more than once with identical values across all columns.

Removing these rows is safe because exact duplicate rows do not add new information and may unnecessarily increase memory usage or affect future joins.

### Decision

The cleaned table `ref_geolocation_clean` will be used for future analysis and joins instead of the original `ref_geolocation`.

Further geolocation checks may still be needed later, especially because the same ZIP code may still appear multiple times with different latitude/longitude values. This is not a full-duplicate issue and should be handled separately if ZIP-level aggregation is required.

## Key-Level Duplicate Check Strategy

After removing exact full duplicate rows from `ref_geolocation`, the next step is to check duplicates at the key level.

High duplicate percentages in individual columns do not always indicate a data quality issue. Many columns are naturally repetitive, such as `order_status`, `payment_type`, `review_score`, city names, states, and category names.

Therefore, the correct approach is to check whether the expected unique identifiers are actually unique.

### Checks to Perform

#### Primary key checks
The following columns are expected to uniquely identify rows:

- `core_customers.customer_id`
- `core_orders.order_id`
- `core_products.product_id`
- `core_sellers.seller_id`
- `ref_category_translation.product_category_name`

#### Composite key checks
Some transactional tables naturally contain repeated IDs. For these tables, composite keys should be checked:

- `core_order_items`: `order_id` + `order_item_id`
- `core_payments`: `order_id` + `payment_sequential`

#### Review table checks
For `core_reviews`, both `review_id` and `order_id` will be checked to understand whether duplicates represent true issues or valid business cases.

In [53]:
# Check whether expected primary keys are unique in their tables.
# This focuses only on columns that should uniquely identify each row.

primary_key_checks = {
    "core_customers": {
        "df": core_customers,
        "key_columns": ["customer_id"]
    },
    "core_orders": {
        "df": core_orders,
        "key_columns": ["order_id"]
    },
    "core_products": {
        "df": core_products,
        "key_columns": ["product_id"]
    },
    "core_sellers": {
        "df": core_sellers,
        "key_columns": ["seller_id"]
    },
    "ref_category_translation": {
        "df": ref_category_translation,
        "key_columns": ["product_category_name"]
    }
}

primary_key_results = []

for table_name, config in primary_key_checks.items():
    df = config["df"]
    key_cols = config["key_columns"]
    
    total_rows = len(df)
    unique_keys = df[key_cols].drop_duplicates().shape[0]
    duplicate_rows = df.duplicated(subset=key_cols).sum()
    
    primary_key_results.append({
        "table": table_name,
        "key_columns": ", ".join(key_cols),
        "total_rows": total_rows,
        "unique_keys": unique_keys,
        "duplicate_key_rows": duplicate_rows,
        "is_key_unique": duplicate_rows == 0
    })

primary_key_results_df = pd.DataFrame(primary_key_results)

print("✅ Primary key uniqueness checks completed")
display(primary_key_results_df)

✅ Primary key uniqueness checks completed


,table,key_columns,total_rows,unique_keys,duplicate_key_rows,is_key_unique
0,core_customers,customer_id,99441,99441,0,True
1,core_orders,order_id,99441,99441,0,True
2,core_products,product_id,32951,32951,0,True
3,core_sellers,seller_id,3095,3095,0,True
4,ref_category_translation,product_category_name,74,74,0,True


In [54]:
# Check composite keys in transactional tables.
# These keys should uniquely identify rows where a single ID may naturally repeat.

composite_key_checks = {
    "core_order_items": {
        "df": core_order_items,
        "key_columns": ["order_id", "order_item_id"]
    },
    "core_payments": {
        "df": core_payments,
        "key_columns": ["order_id", "payment_sequential"]
    }
}

composite_key_results = []

for table_name, config in composite_key_checks.items():
    df = config["df"]
    key_cols = config["key_columns"]
    
    total_rows = len(df)
    unique_keys = df[key_cols].drop_duplicates().shape[0]
    duplicate_rows = df.duplicated(subset=key_cols).sum()
    
    composite_key_results.append({
        "table": table_name,
        "key_columns": ", ".join(key_cols),
        "total_rows": total_rows,
        "unique_keys": unique_keys,
        "duplicate_key_rows": duplicate_rows,
        "is_key_unique": duplicate_rows == 0
    })

composite_key_results_df = pd.DataFrame(composite_key_results)

print("✅ Composite key uniqueness checks completed")
display(composite_key_results_df)

✅ Composite key uniqueness checks completed


,table,key_columns,total_rows,unique_keys,duplicate_key_rows,is_key_unique
0,core_order_items,"order_id, order_item_id",112650,112650,0,True
1,core_payments,"order_id, payment_sequential",103886,103886,0,True


In [55]:
# Diagnose duplicates in the reviews table.
# We check both review_id and order_id to understand the duplication pattern.

review_duplicate_checks = {
    "review_id": ["review_id"],
    "order_id": ["order_id"],
    "review_id_order_id": ["review_id", "order_id"]
}

review_duplicate_results = []

for check_name, key_cols in review_duplicate_checks.items():
    total_rows = len(core_reviews)
    unique_keys = core_reviews[key_cols].drop_duplicates().shape[0]
    duplicate_rows = core_reviews.duplicated(subset=key_cols).sum()
    
    review_duplicate_results.append({
        "check_name": check_name,
        "key_columns": ", ".join(key_cols),
        "total_rows": total_rows,
        "unique_keys": unique_keys,
        "duplicate_key_rows": duplicate_rows,
        "is_key_unique": duplicate_rows == 0
    })

review_duplicate_results_df = pd.DataFrame(review_duplicate_results)

print("✅ Review duplicate diagnosis completed")
display(review_duplicate_results_df)

✅ Review duplicate diagnosis completed


,check_name,key_columns,total_rows,unique_keys,duplicate_key_rows,is_key_unique
0,review_id,review_id,99222,98408,814,False
1,order_id,order_id,99222,98671,551,False
2,review_id_order_id,"review_id, order_id",99222,99222,0,True


In [56]:
# Helper function to inspect duplicated key records.
# This only displays duplicated records for diagnosis. It does not delete anything.

def inspect_duplicated_keys(df, key_cols, table_name, max_rows=50):
    duplicated_records = (
        df[df.duplicated(subset=key_cols, keep=False)]
        .sort_values(key_cols)
        .copy()
    )
    
    print(f"\n===== {table_name} =====")
    print("Key columns:", key_cols)
    print("Duplicated records:", len(duplicated_records))
    
    if len(duplicated_records) > 0:
        display(duplicated_records.head(max_rows))
    else:
        print("✅ No duplicated key records found.")


# Inspect primary key duplicates
for table_name, config in primary_key_checks.items():
    inspect_duplicated_keys(
        df=config["df"],
        key_cols=config["key_columns"],
        table_name=table_name
    )

# Inspect composite key duplicates
for table_name, config in composite_key_checks.items():
    inspect_duplicated_keys(
        df=config["df"],
        key_cols=config["key_columns"],
        table_name=table_name
    )

# Inspect review duplicates
inspect_duplicated_keys(core_reviews, ["review_id"], "core_reviews - review_id")
inspect_duplicated_keys(core_reviews, ["order_id"], "core_reviews - order_id")
inspect_duplicated_keys(core_reviews, ["review_id", "order_id"], "core_reviews - review_id + order_id")


===== core_customers =====
Key columns: ['customer_id']
Duplicated records: 0
✅ No duplicated key records found.

===== core_orders =====
Key columns: ['order_id']
Duplicated records: 0
✅ No duplicated key records found.

===== core_products =====
Key columns: ['product_id']
Duplicated records: 0
✅ No duplicated key records found.

===== core_sellers =====
Key columns: ['seller_id']
Duplicated records: 0
✅ No duplicated key records found.

===== ref_category_translation =====
Key columns: ['product_category_name']
Duplicated records: 0
✅ No duplicated key records found.

===== core_order_items =====
Key columns: ['order_id', 'order_item_id']
Duplicated records: 0
✅ No duplicated key records found.

===== core_payments =====
Key columns: ['order_id', 'payment_sequential']
Duplicated records: 0
✅ No duplicated key records found.

===== core_reviews - review_id =====
Key columns: ['review_id']
Duplicated records: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_row_id,has_review_comment_title,has_review_comment_message,has_any_review_text
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-07-03,2018-03-20 18:08:00,46679,False,True,True
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-07-03,2018-03-20 18:08:00,29842,False,True,True
90675,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:00,90676,False,False,False
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:00,63194,False,False,False
92874,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-07-03,2018-03-08 03:00:00,92875,False,True,True
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-07-03,2018-03-08 03:00:00,57281,False,True,True
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-02-03,2018-03-05 01:43:00,54833,False,False,False
99165,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-02-03,2018-03-05 01:43:00,99166,False,False,False
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:00,20622,False,True,True
96078,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:00,96079,False,True,True



===== core_reviews - order_id =====
Key columns: ['order_id']
Duplicated records: 1098


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_row_id,has_review_comment_title,has_review_comment_message,has_any_review_text
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:00,25613,False,False,False
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:00,22424,False,True,True
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:00,22780,False,False,False
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-04-03,2018-03-05 17:02:00,68634,False,False,False
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:00,855,False,False,False
83222,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:00,83223,False,False,False
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:00,17583,False,True,True
89886,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:00,89887,False,True,True
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:00,55138,False,False,False
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:00,37912,False,False,False



===== core_reviews - review_id + order_id =====
Key columns: ['review_id', 'order_id']
Duplicated records: 0
✅ No duplicated key records found.


## Duplicate Records & Key-Level Duplicate Checks Summary

Duplicate analysis was performed in two levels:

1. **Full row duplicate check**
2. **Key-level duplicate check**

This distinction is important because a duplicated value in a single column does not always mean there is a data quality issue. Some columns are expected to repeat naturally, such as `order_status`, `payment_type`, `review_score`, city names, and state names.

---

## 1. Full Row Duplicate Check

A full duplicate row check was performed across all tables.

### Result

All tables had **0 full duplicate rows** except `ref_geolocation`.

`ref_geolocation` originally contained:

- Original rows: **1,000,163**
- Full duplicate rows removed: **261,831**
- Removed duplicate percentage: **26.18%**
- Cleaned rows: **738,332**
- Remaining full duplicate rows after cleaning: **0**

### Treatment Applied

A cleaned version of the table was created:

`ref_geolocation_clean`

Exact duplicate rows were removed using `drop_duplicates()`, while keeping the original `ref_geolocation` table unchanged.

### Decision

`ref_geolocation_clean` will be used for future analysis and joins instead of the original `ref_geolocation`.

---

## 2. Primary Key Duplicate Checks

Primary key uniqueness checks were performed on the tables where a single column is expected to uniquely identify each row.

### Checked Keys

- `core_customers.customer_id`
- `core_orders.order_id`
- `core_products.product_id`
- `core_sellers.seller_id`
- `ref_category_translation.product_category_name`

### Result

No duplicated key records were found in any of these tables.

This confirms that the main entity tables have valid unique identifiers.

### Decision

No treatment is required for primary keys in these tables.

---

## 3. Composite Key Duplicate Checks

Some transactional tables are expected to have repeated single IDs.  
For example, one order can contain multiple items or multiple payment records.

Therefore, composite keys were checked instead of single-column keys.

### Checked Composite Keys

- `core_order_items`: `order_id` + `order_item_id`
- `core_payments`: `order_id` + `payment_sequential`

### Result

No duplicated composite key records were found.

This means that each order item and each payment sequence is uniquely represented.

### Decision

No treatment is required for composite keys in `core_order_items` or `core_payments`.

---

## 4. Review Table Duplicate Diagnosis

The `core_reviews` table required special investigation because both `review_id` and `order_id` showed duplicated values.

### Results

| Check | Key Columns | Total Rows | Unique Keys | Duplicate Key Rows | Is Key Unique |
|---|---|---:|---:|---:|---|
| Review ID check | `review_id` | 99,222 | 98,408 | 814 | False |
| Order ID check | `order_id` | 99,222 | 98,671 | 551 | False |
| Review ID + Order ID check | `review_id`, `order_id` | 99,222 | 99,222 | 0 | True |

### Important Interpretation

Although `review_id` alone is duplicated and `order_id` alone is duplicated, the combination of:

`review_id` + `order_id`

is fully unique.

This means there are no duplicated review-order pairs.

### Why the inspected duplicated record counts are higher

During inspection:

- `review_id` duplicated records displayed: **1,603**
- `order_id` duplicated records displayed: **1,098**

These counts are higher than `duplicate_key_rows` because the inspection used `keep=False`, which shows **all records involved in duplication**, including the first occurrence.

Meanwhile, `duplicate_key_rows` counts only the repeated rows after the first occurrence.

So this difference is expected and does not indicate a problem.

---

## Final Duplicate Treatment Decision

### What was fixed

- Exact full duplicate rows were removed only from `ref_geolocation`.
- A cleaned table `ref_geolocation_clean` was created and verified.

### What was not removed

No records were removed from:

- `core_customers`
- `core_orders`
- `core_products`
- `core_sellers`
- `core_order_items`
- `core_payments`
- `core_reviews`
- `ref_category_translation`

### Why no review records were removed

The `core_reviews` table does not contain duplicated `review_id` + `order_id` combinations.

Even though `review_id` and `order_id` appear duplicated individually, the combined key is unique. Therefore, the duplicated individual IDs likely represent valid review relationships rather than exact duplicate records.

---

## Conclusion

The duplicate issue is considered resolved.

- Full duplicate rows were found only in `ref_geolocation` and were successfully removed.
- Primary keys in the main entity tables are unique.
- Composite keys in transactional tables are unique.
- Review records are unique at the `review_id` + `order_id` level.
- No additional duplicate removal is required at this stage.

Further geolocation preparation may still be needed later because the same ZIP code may appear multiple times with different latitude/longitude values. This is a separate geolocation aggregation issue, not a full duplicate row issue.

## Geolocation ZIP-Level Duplicate Treatment

After removing exact full duplicate rows from `ref_geolocation`, the cleaned table `ref_geolocation_clean` may still contain multiple records for the same `geolocation_zip_code_prefix`.

This is not a full duplicate issue. It means that the same ZIP code can appear with different latitude, longitude, city, or state values.

### Why this matters

The geolocation table will later be joined with customer and seller tables using ZIP code fields:

- `core_customers.customer_zip_code_prefix`
- `core_sellers.seller_zip_code_prefix`

If one ZIP code appears multiple times in the geolocation table, joins may duplicate customer or seller rows and inflate analytical results.

### Treatment Decision

A new ZIP-level lookup table will be created:

`ref_geolocation_zip`

This table will contain one row per ZIP code using:

- average latitude,
- average longitude,
- most frequent city,
- most frequent state.

The original `ref_geolocation_clean` table will remain unchanged.

In [57]:
# Diagnose ZIP-level duplicates in ref_geolocation_clean.
# This checks whether the same ZIP code appears multiple times.

import pandas as pd

zip_col = "geolocation_zip_code_prefix"

total_geo_rows = len(ref_geolocation_clean)
unique_zip_codes = ref_geolocation_clean[zip_col].nunique()
extra_rows_due_to_zip_duplicates = total_geo_rows - unique_zip_codes

zip_counts = (
    ref_geolocation_clean[zip_col]
    .value_counts()
    .reset_index()
)

zip_counts.columns = [zip_col, "row_count"]

duplicated_zip_codes_count = (zip_counts["row_count"] > 1).sum()
max_rows_per_zip = zip_counts["row_count"].max()

zip_duplicate_summary = pd.DataFrame({
    "metric": [
        "Total rows in ref_geolocation_clean",
        "Unique ZIP codes",
        "Extra rows due to ZIP-level duplicates",
        "ZIP codes appearing more than once",
        "Maximum rows for a single ZIP code"
    ],
    "value": [
        total_geo_rows,
        unique_zip_codes,
        extra_rows_due_to_zip_duplicates,
        duplicated_zip_codes_count,
        max_rows_per_zip
    ]
})

display(zip_duplicate_summary)

,metric,value
0,Total rows in ref_geolocation_clean,738332
1,Unique ZIP codes,19015
2,Extra rows due to ZIP-level duplicates,719317
3,ZIP codes appearing more than once,17823
4,Maximum rows for a single ZIP code,779


In [58]:
# Show the most repeated ZIP codes.
# This helps understand the scale of ZIP-level duplication.

top_repeated_zips = zip_counts[zip_counts["row_count"] > 1].head(20)

display(top_repeated_zips)

,geolocation_zip_code_prefix,row_count
0,38400,779
1,35500,751
2,11680,727
3,11740,678
4,36400,627
5,38408,621
6,39400,620
7,35162,611
8,37200,596
9,35900,589


In [59]:
# Inspect all geolocation rows for the most repeated ZIP code.

most_repeated_zip = top_repeated_zips.iloc[0][zip_col]

print("Most repeated ZIP code:", most_repeated_zip)

display(
    ref_geolocation_clean[
        ref_geolocation_clean[zip_col] == most_repeated_zip
    ].head(30)
)

Most repeated ZIP code: 38400


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
645450,38400,-18.915960,-48.278974,uberlandia,MG
645456,38400,-18.900442,-48.260759,uberlandia,MG
645460,38400,-18.913015,-48.262132,uberlandia,MG
645461,38400,-18.922381,-48.282111,uberlandia,MG
645471,38400,-18.922160,-48.271125,uberlandia,MG
645474,38400,-18.915501,-48.263653,uberlândia,MG
645479,38400,-18.921864,-48.295130,uberlandia,MG
645494,38400,-18.903228,-48.274547,uberlândia,MG
645501,38400,-18.903908,-48.273556,uberlandia,MG
645511,38400,-18.923919,-48.278409,uberlandia,MG


In [60]:
# Helper function to get the most frequent value.
# If there is a tie, the first mode value will be selected.

def most_frequent_value(series):
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return None
    return mode_values.iloc[0]


# Create a ZIP-level geolocation lookup table.
# Each ZIP code will appear only once.

ref_geolocation_zip = (
    ref_geolocation_clean
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", most_frequent_value),
        geolocation_state=("geolocation_state", most_frequent_value)
    )
)

print("✅ ZIP-level geolocation table created")
print("Rows in ref_geolocation_clean:", len(ref_geolocation_clean))
print("Rows in ref_geolocation_zip:", len(ref_geolocation_zip))

display(ref_geolocation_zip.head())

✅ ZIP-level geolocation table created
Rows in ref_geolocation_clean: 738332
Rows in ref_geolocation_zip: 19015


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550227,-46.634039,sao paulo,SP
1,1002,-23.547657,-46.634991,sao paulo,SP
2,1003,-23.549000,-46.635582,sao paulo,SP
3,1004,-23.549829,-46.634792,sao paulo,SP
4,1005,-23.549547,-46.636406,sao paulo,SP


In [61]:
# Verify that each ZIP code appears only once in ref_geolocation_zip.

zip_duplicate_after_aggregation = ref_geolocation_zip.duplicated(
    subset=["geolocation_zip_code_prefix"]
).sum()

print("Duplicate ZIP codes in ref_geolocation_zip:", zip_duplicate_after_aggregation)

if zip_duplicate_after_aggregation == 0:
    print("✅ ref_geolocation_zip has one row per ZIP code.")
else:
    print("⚠️ Some ZIP codes are still duplicated and need investigation.")

Duplicate ZIP codes in ref_geolocation_zip: 0
✅ ref_geolocation_zip has one row per ZIP code.


In [62]:
# Check whether joining customers with ref_geolocation_zip preserves row count.

customers_geo_test = core_customers.merge(
    ref_geolocation_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

print("Customers rows before join:", len(core_customers))
print("Customers rows after join:", len(customers_geo_test))


# Check whether joining sellers with ref_geolocation_zip preserves row count.

sellers_geo_test = core_sellers.merge(
    ref_geolocation_zip,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

print("Sellers rows before join:", len(core_sellers))
print("Sellers rows after join:", len(sellers_geo_test))

Customers rows before join: 99441
Customers rows after join: 99441
Sellers rows before join: 3095
Sellers rows after join: 3095


In [63]:
# Check customers without matched geolocation.

customers_without_geo = customers_geo_test["geolocation_lat"].isna().sum()
customers_without_geo_pct = round(customers_without_geo / len(customers_geo_test) * 100, 2)

print("Customers without geolocation match:", customers_without_geo)
print("Customers without geolocation match %:", customers_without_geo_pct)


# Check sellers without matched geolocation.

sellers_without_geo = sellers_geo_test["geolocation_lat"].isna().sum()
sellers_without_geo_pct = round(sellers_without_geo / len(sellers_geo_test) * 100, 2)

print("Sellers without geolocation match:", sellers_without_geo)
print("Sellers without geolocation match %:", sellers_without_geo_pct)

Customers without geolocation match: 278
Customers without geolocation match %: 0.28
Sellers without geolocation match: 7
Sellers without geolocation match %: 0.23


In [64]:
# Export the final ZIP-level geolocation table to CSV.
# This file will be used for future analysis and joins.

ref_geolocation_zip.to_csv(
    "ref_geolocation_zip.csv",
    index=False,
    encoding="utf-8"
)

print("✅ ref_geolocation_zip.csv has been created successfully")

✅ ref_geolocation_zip.csv has been created successfully


## Geolocation ZIP-Level Duplicate Treatment Summary

After removing exact full duplicate rows from `ref_geolocation`, the cleaned table `ref_geolocation_clean` still contained multiple records for the same ZIP code.

This was a ZIP-level duplication issue, not a full-row duplication issue.

### Diagnosis Results

The ZIP-level duplicate analysis showed:

- Rows in `ref_geolocation_clean`: **738,332**
- Unique ZIP codes: **19,015**
- Extra rows due to ZIP-level duplicates: **719,317**
- ZIP codes appearing more than once: **17,823**
- Maximum rows for a single ZIP code: **779**

This means that many ZIP codes appeared multiple times with different latitude, longitude, city, or state values.

### Example

The most repeated ZIP code was:

- ZIP code: **38400**
- Number of rows: **779**

Inspection showed that this ZIP code appeared many times with slightly different latitude and longitude values, while mostly referring to the same city/state area.

### Why This Matters

If `ref_geolocation_clean` were joined directly with customer or seller tables, one ZIP code could match multiple geolocation rows.

This could duplicate customer or seller records during joins and inflate analysis results.

For example, if one customer ZIP code appears 10 times in the geolocation table, the customer row could appear 10 times after the join.

### Treatment Applied

A new ZIP-level lookup table was created:

`ref_geolocation_zip`

This table contains one row per `geolocation_zip_code_prefix`.

The aggregation logic used was:

- Mean latitude
- Mean longitude
- Most frequent city
- Most frequent state

### Validation Results

After aggregation:

- Duplicate ZIP codes in `ref_geolocation_zip`: **0**
- `ref_geolocation_zip` has exactly one row per ZIP code.

Join safety checks were also performed:

#### Customers Join Check

- Customers rows before join: **99,441**
- Customers rows after join: **99,441**

This confirms that joining customers with `ref_geolocation_zip` does not inflate customer rows.

#### Sellers Join Check

- Sellers rows before join: **3,095**
- Sellers rows after join: **3,095**

This confirms that joining sellers with `ref_geolocation_zip` does not inflate seller rows.

### Missing Geolocation After Join

Some customers and sellers still did not find a matching ZIP code in the geolocation lookup table:

- Customers without geolocation match: **278**
- Customers without geolocation match percentage: **0.28%**
- Sellers without geolocation match: **7**
- Sellers without geolocation match percentage: **0.23%**

These unmatched records are very small percentages and should be flagged or handled during EDA if geographic analysis is required.

### Output

The final ZIP-level geolocation lookup table was exported successfully as:

`ref_geolocation_zip.csv`

### Final Decision

The ZIP-level duplicate issue is considered resolved.

For future analysis and joins, `ref_geolocation_zip` should be used instead of `ref_geolocation_clean` when joining geolocation data with customers or sellers.

The original `ref_geolocation_clean` remains available as the cleaned but non-aggregated geolocation table.

# Problem No.5:
## Referential Integrity Checks

Referential integrity checks are used to confirm that foreign keys in child tables have valid matching keys in their related parent tables.

This is important because missing parent records can cause failed joins, missing values after merging, row loss, or inaccurate analysis.

### Relationships Checked

The following relationships will be validated:

- `core_orders.customer_id` → `core_customers.customer_id`
- `core_order_items.order_id` → `core_orders.order_id`
- `core_order_items.product_id` → `core_products.product_id`
- `core_order_items.seller_id` → `core_sellers.seller_id`
- `core_payments.order_id` → `core_orders.order_id`
- `core_reviews.order_id` → `core_orders.order_id`
- `core_products.product_category_name` → `ref_category_translation.product_category_name`
- `core_customers.customer_zip_code_prefix` → `ref_geolocation_zip.geolocation_zip_code_prefix`
- `core_sellers.seller_zip_code_prefix` → `ref_geolocation_zip.geolocation_zip_code_prefix`

### Decision

Any unmatched child keys will be identified and inspected before deciding whether they should be flagged, excluded, or handled during EDA.

In [65]:
import pandas as pd

def check_referential_integrity(
    child_df,
    parent_df,
    child_key,
    parent_key,
    relationship_name
):
    """
    Check referential integrity between a child table and a parent table.

    The function checks whether every non-null key in the child table
    exists in the parent table.

    Outputs:
    - total child rows
    - unique child keys
    - unmatched child keys
    - unmatched child rows
    - unmatched percentages
    """

    # Get non-null keys from child and parent tables
    child_keys = child_df[child_key].dropna()
    parent_keys = parent_df[parent_key].dropna()

    # Convert keys to sets for comparison
    child_key_set = set(child_keys.unique())
    parent_key_set = set(parent_keys.unique())

    # Identify keys that exist in child but not in parent
    unmatched_keys = child_key_set - parent_key_set

    # Count child rows affected by unmatched keys
    unmatched_rows = child_df[child_df[child_key].isin(unmatched_keys)]

    result = {
        "relationship": relationship_name,
        "child_key": child_key,
        "parent_key": parent_key,
        "child_total_rows": len(child_df),
        "child_unique_keys": len(child_key_set),
        "parent_unique_keys": len(parent_key_set),
        "unmatched_unique_keys": len(unmatched_keys),
        "unmatched_child_rows": len(unmatched_rows),
        "unmatched_unique_keys_pct": round(len(unmatched_keys) / len(child_key_set) * 100, 4) if len(child_key_set) > 0 else 0,
        "unmatched_child_rows_pct": round(len(unmatched_rows) / len(child_df) * 100, 4) if len(child_df) > 0 else 0
    }

    return result, unmatched_keys, unmatched_rows

In [69]:
# Define all referential integrity checks.
# Eachity_results_df)# Each check compares a child foreign key against a parent key.


relationship_checks = [
    {
        "relationship_name": "orders -> customers",
        "child_df": core_orders,
        "parent_df": core_customers,
        "child_key": "customer_id",
        "parent_key": "customer_id"
    },
    {
        "relationship_name": "order_items -> orders",
        "child_df": core_order_items,
        "parent_df": core_orders,
        "child_key": "order_id",
        "parent_key": "order_id"
    },
    {
        "relationship_name": "order_items -> products",
        "child_df": core_order_items,
        "parent_df": core_products,
        "child_key": "product_id",
        "parent_key": "product_id"
    },
    {
        "relationship_name": "order_items -> sellers",
        "child_df": core_order_items,
        "parent_df": core_sellers,
        "child_key": "seller_id",
        "parent_key": "seller_id"
    },
    {
        "relationship_name": "payments -> orders",
        "child_df": core_payments,
        "parent_df": core_orders,
        "child_key": "order_id",
        "parent_key": "order_id"
    },
    {
        "relationship_name": "reviews -> orders",
        "child_df": core_reviews,
        "parent_df": core_orders,
        "child_key": "order_id",
        "parent_key": "order_id"
    },
    {
        "relationship_name": "products -> category_translation",
        "child_df": core_products,
        "parent_df": ref_category_translation,
        "child_key": "product_category_name",
        "parent_key": "product_category_name"
    },
    {
        "relationship_name": "customers -> geolocation_zip",
        "child_df": core_customers,
        "parent_df": ref_geolocation_zip,
        "child_key": "customer_zip_code_prefix",
        "parent_key": "geolocation_zip_code_prefix"
    },
    {
        "relationship_name": "sellers -> geolocation_zip",
        "child_df": core_sellers,
        "parent_df": ref_geolocation_zip,
        "child_key": "seller_zip_code_prefix",
        "parent_key": "geolocation_zip_code_prefix"
    }
]

# Run all checks and store results
ri_results = []
ri_unmatched_keys = {}
ri_unmatched_rows = {}

for check in relationship_checks:
    result, unmatched_keys, unmatched_rows = check_referential_integrity(
        child_df=check["child_df"],
        parent_df=check["parent_df"],
        child_key=check["child_key"],
        parent_key=check["parent_key"],
        relationship_name=check["relationship_name"]
    )

    ri_results.append(result)
    ri_unmatched_keys[check["relationship_name"]] = unmatched_keys
    ri_unmatched_rows[check["relationship_name"]] = unmatched_rows

referential_integrity_results_df = pd.DataFrame(ri_results)

print("✅ Referential integrity checks completed")
display(referential_integrity_results_df)


✅ Referential integrity checks completed


,relationship,child_key,parent_key,child_total_rows,child_unique_keys,parent_unique_keys,unmatched_unique_keys,unmatched_child_rows,unmatched_unique_keys_pct,unmatched_child_rows_pct
0,orders -> customers,customer_id,customer_id,99441,99441,99441,0,0,0.0000,0.0000
1,order_items -> orders,order_id,order_id,112650,98666,99441,0,0,0.0000,0.0000
2,order_items -> products,product_id,product_id,112650,32951,32951,0,0,0.0000,0.0000
3,order_items -> sellers,seller_id,seller_id,112650,3095,3095,0,0,0.0000,0.0000
4,payments -> orders,order_id,order_id,103886,99440,99441,0,0,0.0000,0.0000
5,reviews -> orders,order_id,order_id,99222,98671,99441,0,0,0.0000,0.0000
6,products -> category_translation,product_category_name,product_category_name,32951,74,74,0,0,0.0000,0.0000
7,customers -> geolocation_zip,customer_zip_code_prefix,geolocation_zip_code_prefix,99441,14994,19015,157,278,1.0471,0.2796
8,sellers -> geolocation_zip,seller_zip_code_prefix,geolocation_zip_code_prefix,3095,2246,19015,7,7,0.3117,0.2262


In [70]:
# Filter relationships that have unmatched child rows.
# These relationships need inspection or treatment.

ri_failed_checks = referential_integrity_results_df[
    referential_integrity_results_df["unmatched_child_rows"] > 0
].copy()

print("Number of relationships with unmatched child rows:", len(ri_failed_checks))

if len(ri_failed_checks) > 0:
    display(ri_failed_checks)
else:
    print("✅ All checked relationships passed referential integrity.")

Number of relationships with unmatched child rows: 2


,relationship,child_key,parent_key,child_total_rows,child_unique_keys,parent_unique_keys,unmatched_unique_keys,unmatched_child_rows,unmatched_unique_keys_pct,unmatched_child_rows_pct
7,customers -> geolocation_zip,customer_zip_code_prefix,geolocation_zip_code_prefix,99441,14994,19015,157,278,1.0471,0.2796
8,sellers -> geolocation_zip,seller_zip_code_prefix,geolocation_zip_code_prefix,3095,2246,19015,7,7,0.3117,0.2262


In [71]:
# Inspect unmatched rows for each failed relationship.
# This helps understand whether unmatched keys are major issues or small exceptions.

for relationship_name, unmatched_df in ri_unmatched_rows.items():
    if len(unmatched_df) > 0:
        print(f"\n===== {relationship_name} =====")
        print("Unmatched rows:", len(unmatched_df))
        display(unmatched_df.head(20))


===== customers -> geolocation_zip =====
Unmatched rows: 278


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
1483,03bbe0ce5c28e05f22917607db798818,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF
3045,07e37b9181238afc5aef3d87171bf28f,aa671b18d12bf808c52e3ce9875dd8c6,71551,brasilia,DF
3553,093afa5e2f16785b06a54179fdc986c2,b6e4d8aa8df3bd5ce682be0ef338e5da,71919,brasilia,DF
3666,0989ad22ee10df514ada88585ebd3101,37d072bb642db6424d5b3693de406a4f,37005,varginha,MG
4136,0abda7ee9b75764b3bf4197b63ee7c0a,a28a5bf286affa0abc5f7dbc1ccb6a2d,73255,brasilia,DF
4820,0c7d17d2ae9a0c85566d9d58ea478a25,f6a1b401ed3ffaa8ed4de1a6da5079b0,72005,brasilia,DF
5255,0da03c17bb6e78ca54c42dd0332b5847,6a8733ff8b8426816669b13aa1f07501,29196,aracruz,ES
5357,0debfbe6eb17e95af641df3e543d5959,b430e3d118c6f0ba3a07a99b6b9222bf,71996,brasilia,DF
5422,0e1b17d09c043febb1b71ade300fc357,592f7582fefbe45dd0986a1fbbf80e13,57254,luziapolis,AL
5620,0e9c3be1592c050d343700ba21b7f806,1c39144d899f035157a7761e319bc3ea,71676,brasilia,DF



===== sellers -> geolocation_zip =====
Unmatched rows: 7


,seller_id,seller_zip_code_prefix,seller_city,seller_state
144,0b3f27369a4d8df98f7eb91077e438ac,7412,aruja,SP
495,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,brasilia,DF
504,2aafae69bf4c41fbd94053d9413e87ee,91901,porto alegre,RS
552,2e90cb1677d35cfe24eef47d441b7c87,2285,sao paulo,SP
796,42bde9fef835393bb8a8849cb6b7f245,71551,brasilia,DF
1088,5962468f885ea01a1b6a97a218797b0a,82040,curitiba,PR
1655,870d0118f7a9d85960f29ad89d5d989a,37708,pocos de caldas,MG


In [72]:
# Create geolocation match flags for customers and sellers.
# These flags show whether each customer/seller ZIP code exists in ref_geolocation_zip.

valid_geo_zips = set(ref_geolocation_zip["geolocation_zip_code_prefix"].dropna().unique())

core_customers["has_geolocation_match"] = (
    core_customers["customer_zip_code_prefix"].isin(valid_geo_zips)
)

core_sellers["has_geolocation_match"] = (
    core_sellers["seller_zip_code_prefix"].isin(valid_geo_zips)
)

customer_geo_match_summary = core_customers["has_geolocation_match"].value_counts(dropna=False).reset_index()
customer_geo_match_summary.columns = ["has_geolocation_match", "customer_count"]
customer_geo_match_summary["percentage"] = (
    customer_geo_match_summary["customer_count"] / len(core_customers) * 100
).round(2)

seller_geo_match_summary = core_sellers["has_geolocation_match"].value_counts(dropna=False).reset_index()
seller_geo_match_summary.columns = ["has_geolocation_match", "seller_count"]
seller_geo_match_summary["percentage"] = (
    seller_geo_match_summary["seller_count"] / len(core_sellers) * 100
).round(2)

print("✅ Geolocation match flags created")

print("\nCustomer geolocation match summary:")
display(customer_geo_match_summary)

print("\nSeller geolocation match summary:")
display(seller_geo_match_summary)

✅ Geolocation match flags created

Customer geolocation match summary:


,has_geolocation_match,customer_count,percentage
0,True,99163,99.72
1,False,278,0.28



Seller geolocation match summary:


,has_geolocation_match,seller_count,percentage
0,True,3088,99.77
1,False,7,0.23


## Referential Integrity Check Summary

Referential integrity checks were performed to verify that foreign keys in child tables have valid matching keys in their related parent tables.

This step helps confirm that joins between tables will not fail due to missing parent records.

---

### Relationships Checked

The following relationships were validated:

- `core_orders.customer_id` → `core_customers.customer_id`
- `core_order_items.order_id` → `core_orders.order_id`
- `core_order_items.product_id` → `core_products.product_id`
- `core_order_items.seller_id` → `core_sellers.seller_id`
- `core_payments.order_id` → `core_orders.order_id`
- `core_reviews.order_id` → `core_orders.order_id`
- `core_products.product_category_name` → `ref_category_translation.product_category_name`
- `core_customers.customer_zip_code_prefix` → `ref_geolocation_zip.geolocation_zip_code_prefix`
- `core_sellers.seller_zip_code_prefix` → `ref_geolocation_zip.geolocation_zip_code_prefix`

---

### Passed Relationships

The following relationships passed with **0 unmatched child rows**:

- `orders -> customers`
- `order_items -> orders`
- `order_items -> products`
- `order_items -> sellers`
- `payments -> orders`
- `reviews -> orders`
- `products -> category_translation`

This means that all foreign keys in these child tables have valid matching records in their parent tables.

---

### Relationships with Unmatched Records

Two geolocation-related relationships had unmatched records:

#### `customers -> geolocation_zip`

- Child key: `customer_zip_code_prefix`
- Parent key: `geolocation_zip_code_prefix`
- Customer rows: **99,441**
- Unique customer ZIP codes: **14,994**
- Unique geolocation ZIP codes: **19,015**
- Unmatched unique ZIP codes: **157**
- Unmatched customer rows: **278**
- Unmatched unique ZIP percentage: **1.0471%**
- Unmatched customer row percentage: **0.2796%**

#### `sellers -> geolocation_zip`

- Child key: `seller_zip_code_prefix`
- Parent key: `geolocation_zip_code_prefix`
- Seller rows: **3,095**
- Unique seller ZIP codes: **2,246**
- Unique geolocation ZIP codes: **19,015**
- Unmatched unique ZIP codes: **7**
- Unmatched seller rows: **7**
- Unmatched unique ZIP percentage: **0.3117%**
- Unmatched seller row percentage: **0.2262%**

---

### Geolocation Match Flags

Because the only failed relationships were related to geolocation ZIP matching, records were not removed.

Instead, geolocation match flags were created:

- `core_customers.has_geolocation_match`
- `core_sellers.has_geolocation_match`

These flags identify whether each customer or seller ZIP code exists in `ref_geolocation_zip`.

---

### Geolocation Match Results

#### Customers

- Customers with geolocation match: **99,163** (**99.72%**)
- Customers without geolocation match: **278** (**0.28%**)

#### Sellers

- Sellers with geolocation match: **3,088** (**99.77%**)
- Sellers without geolocation match: **7** (**0.23%**)

---

### Interpretation

All core transactional and master-data relationships passed the referential integrity checks.

The only unmatched records were related to ZIP code matching with the geolocation lookup table.

The unmatched geolocation percentages are very small:

- **0.28%** of customers
- **0.23%** of sellers

This indicates that the issue is limited and should not affect the overall dataset significantly.

---

### Treatment Decision

- No records were deleted.
- Core relational links between orders, customers, products, sellers, payments, reviews, and category translation are valid.
- Geolocation unmatched records were flagged instead of removed.
- The flags will be used later during geographic EDA to decide whether unmatched records should be excluded from maps or analyzed separately.

---

### Final Decision

The referential integrity issue is considered resolved.

For future analysis:

- Use `ref_geolocation_zip` for geolocation joins.
- Use `has_geolocation_match` flags to filter or identify records without matched geolocation data.
- Avoid dropping unmatched customer or seller records unless the analysis specifically requires geographic coordinates.

# Problem No. 6
## Numeric Value Validation

Numeric validation checks are performed to identify invalid or unrealistic numeric values.

A column can have the correct numeric data type but still contain values that do not make business sense.

### Columns Checked

#### Order Items
- `price <= 0`
- `freight_value < 0`

#### Payments
- `payment_value <= 0`
- `payment_installments <= 0`

#### Products
- `product_weight_g <= 0`
- `product_length_cm <= 0`
- `product_height_cm <= 0`
- `product_width_cm <= 0`
- `product_photos_qty < 0`

### Treatment Decision

At this stage, invalid numeric values will not be removed automatically.

They will be flagged and inspected first to determine whether they represent true data errors, valid edge cases, or values that should be handled later during EDA or modeling.
``

In [73]:
# Define numeric validation rules.
# Each rule checks for values that are not expected based on business logic.

numeric_validation_rules = [
    {
        "table": "core_order_items",
        "df": core_order_items,
        "column": "price",
        "condition": lambda df: df["price"] <= 0,
        "issue": "price <= 0"
    },
    {
        "table": "core_order_items",
        "df": core_order_items,
        "column": "freight_value",
        "condition": lambda df: df["freight_value"] < 0,
        "issue": "freight_value < 0"
    },
    {
        "table": "core_payments",
        "df": core_payments,
        "column": "payment_value",
        "condition": lambda df: df["payment_value"] <= 0,
        "issue": "payment_value <= 0"
    },
    {
        "table": "core_payments",
        "df": core_payments,
        "column": "payment_installments",
        "condition": lambda df: df["payment_installments"] <= 0,
        "issue": "payment_installments <= 0"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_weight_g",
        "condition": lambda df: df["product_weight_g"] <= 0,
        "issue": "product_weight_g <= 0"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_length_cm",
        "condition": lambda df: df["product_length_cm"] <= 0,
        "issue": "product_length_cm <= 0"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_height_cm",
        "condition": lambda df: df["product_height_cm"] <= 0,
        "issue": "product_height_cm <= 0"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_width_cm",
        "condition": lambda df: df["product_width_cm"] <= 0,
        "issue": "product_width_cm <= 0"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_photos_qty",
        "condition": lambda df: df["product_photos_qty"] < 0,
        "issue": "product_photos_qty < 0"
    }
]

In [76]:
# Run numeric validation checks and summarize invalid values.

numeric_validation_summary = []
numeric_invalid_rows = {}

for rule in numeric_validation_rules:
    table_name = rule["table"]
    df = rule["df"]
    column = rule["column"]
    issue = rule["issue"]
    
    # Apply the validation condition
    invalid_mask = rule["condition"](df)
    invalid_rows = df[invalid_mask].copy()
    
    # Store invalid rows for later inspection
    key = f"{table_name}.{column} - {issue}"
    numeric_invalid_rows[key] = invalid_rows
    
    numeric_validation_summary.append({
        "table": table_name,
        "column": column,
        "issue": issue,
        "total_rows": len(df),
        "invalid_rows": len(invalid_rows),
        "invalid_percentage": round(len(invalid_rows) / len(df) * 100, 4)
    })

numeric_validation_summary_df = pd.DataFrame(numeric_validation_summary)

print("✅ Numeric validation checks completed")
display(numeric_validation_summary_df)


✅ Numeric validation checks completed


,table,column,issue,total_rows,invalid_rows,invalid_percentage
0,core_order_items,price,price <= 0,112650,0,0.0000
1,core_order_items,freight_value,freight_value < 0,112650,0,0.0000
2,core_payments,payment_value,payment_value <= 0,103886,9,0.0087
3,core_payments,payment_installments,payment_installments <= 0,103886,2,0.0019
4,core_products,product_weight_g,product_weight_g <= 0,32951,6,0.0182
5,core_products,product_length_cm,product_length_cm <= 0,32951,2,0.0061
6,core_products,product_height_cm,product_height_cm <= 0,32951,2,0.0061
7,core_products,product_width_cm,product_width_cm <= 0,32951,2,0.0061
8,core_products,product_photos_qty,product_photos_qty < 0,32951,0,0.0000


In [78]:
# Show only numeric validation checks that found invalid rows.

numeric_issues_found = numeric_validation_summary_df[
    numeric_validation_summary_df["invalid_rows"] > 0
].copy()

print("Number of numeric validation issues found:", len(numeric_issues_found))

if len(numeric_issues_found) > 0:
    display(numeric_issues_found)
else:
    print("✅ No invalid numeric values found based on the defined rules.")

Number of numeric validation issues found: 6


,table,column,issue,total_rows,invalid_rows,invalid_percentage
2,core_payments,payment_value,payment_value <= 0,103886,9,0.0087
3,core_payments,payment_installments,payment_installments <= 0,103886,2,0.0019
4,core_products,product_weight_g,product_weight_g <= 0,32951,6,0.0182
5,core_products,product_length_cm,product_length_cm <= 0,32951,2,0.0061
6,core_products,product_height_cm,product_height_cm <= 0,32951,2,0.0061
7,core_products,product_width_cm,product_width_cm <= 0,32951,2,0.0061


In [79]:
# Inspect invalid rows for each numeric validation issue.
# This is for diagnosis only. No records are modified or deleted here.

for issue_name, invalid_df in numeric_invalid_rows.items():
    if len(invalid_df) > 0:
        print(f"\n===== {issue_name} =====")
        print("Invalid rows:", len(invalid_df))
        display(invalid_df.head(20))


===== core_payments.payment_value - payment_value <= 0 =====
Invalid rows: 9


,order_id,payment_sequential,payment_type,payment_installments,payment_value
259,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
28436,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
28566,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
44305,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
56620,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
72173,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0
81460,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
101581,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
101582,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0



===== core_payments.payment_installments - payment_installments <= 0 =====
Invalid rows: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
10691,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
47346,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69



===== core_products.product_weight_g - product_weight_g <= 0 =====
Invalid rows: 6


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1275,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,0,0,0,0
12265,5eb564652db742ff8f28759cd8d2652a,unknown,0,0,0,0,0,0,0



===== core_products.product_height_cm - product_height_cm <= 0 =====
Invalid rows: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1275,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,0,0,0,0
12265,5eb564652db742ff8f28759cd8d2652a,unknown,0,0,0,0,0,0,0



===== core_products.product_width_cm - product_width_cm <= 0 =====
Invalid rows: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1275,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,0,0,0,0
12265,5eb564652db742ff8f28759cd8d2652a,unknown,0,0,0,0,0,0,0


In [81]:
# Create numeric data quality flags.
# These flags preserve original values while marking invalid numeric records.

# core_order_items flags
core_order_items["dq_price_non_positive"] = core_order_items["price"] <= 0
core_order_items["dq_freight_negative"] = core_order_items["freight_value"] < 0

# core_payments flags
core_payments["dq_payment_value_non_positive"] = core_payments["payment_value"] <= 0
core_payments["dq_payment_installments_non_positive"] = core_payments["payment_installments"] <= 0

# core_products flags
core_products["dq_product_weight_non_positive"] = core_products["product_weight_g"] <= 0
core_products["dq_product_length_non_positive"] = core_products["product_length_cm"] <= 0
core_products["dq_product_height_non_positive"] = core_products["product_height_cm"] <= 0
core_products["dq_product_width_non_positive"] = core_products["product_width_cm"] <= 0
core_products["dq_product_photos_negative"] = core_products["product_photos_qty"] < 0

print("✅ Numeric data quality flags created")

✅ Numeric data quality flags created


In [82]:
# Summarize numeric data quality flags across affected tables.

numeric_flag_summary = []

flag_configs = [
    ("core_order_items", core_order_items, [
        "dq_price_non_positive",
        "dq_freight_negative"
    ]),
    ("core_payments", core_payments, [
        "dq_payment_value_non_positive",
        "dq_payment_installments_non_positive"
    ]),
    ("core_products", core_products, [
        "dq_product_weight_non_positive",
        "dq_product_length_non_positive",
        "dq_product_height_non_positive",
        "dq_product_width_non_positive",
        "dq_product_photos_negative"
    ])
]

for table_name, df, flags in flag_configs:
    for flag in flags:
        numeric_flag_summary.append({
            "table": table_name,
            "flag": flag,
            "flagged_rows": df[flag].sum(),
            "flagged_percentage": round(df[flag].sum() / len(df) * 100, 4)
        })

numeric_flag_summary_df = pd.DataFrame(numeric_flag_summary)

display(numeric_flag_summary_df)

,table,flag,flagged_rows,flagged_percentage
0,core_order_items,dq_price_non_positive,0,0.0000
1,core_order_items,dq_freight_negative,0,0.0000
2,core_payments,dq_payment_value_non_positive,9,0.0087
3,core_payments,dq_payment_installments_non_positive,2,0.0019
4,core_products,dq_product_weight_non_positive,6,0.0182
5,core_products,dq_product_length_non_positive,2,0.0061
6,core_products,dq_product_height_non_positive,2,0.0061
7,core_products,dq_product_width_non_positive,2,0.0061
8,core_products,dq_product_photos_negative,0,0.0000


## Numeric Value Validation Summary

Numeric value validation was performed to identify values that are technically numeric but do not make business sense.

The validation focused on prices, payment values, installments, product weight, product dimensions, and product photo quantities.

---

### Validation Rules Applied

The following rules were checked:

#### Order Items
- `price <= 0`
- `freight_value < 0`

#### Payments
- `payment_value <= 0`
- `payment_installments <= 0`

#### Products
- `product_weight_g <= 0`
- `product_length_cm <= 0`
- `product_height_cm <= 0`
- `product_width_cm <= 0`
- `product_photos_qty < 0`

---

### Issues Found

A total of **6 numeric validation issues** were found.

| Table | Column | Issue | Total Rows | Invalid Rows | Invalid % |
|---|---|---|---:|---:|---:|
| `core_payments` | `payment_value` | `payment_value <= 0` | 103,886 | 9 | 0.0087% |
| `core_payments` | `payment_installments` | `payment_installments <= 0` | 103,886 | 2 | 0.0019% |
| `core_products` | `product_weight_g` | `product_weight_g <= 0` | 32,951 | 6 | 0.0182% |
| `core_products` | `product_length_cm` | `product_length_cm <= 0` | 32,951 | 2 | 0.0061% |
| `core_products` | `product_height_cm` | `product_height_cm <= 0` | 32,951 | 2 | 0.0061% |
| `core_products` | `product_width_cm` | `product_width_cm <= 0` | 32,951 | 2 | 0.0061% |

---

### Checks With No Issues

The following numeric checks returned **0 invalid rows**:

| Table | Check | Invalid Rows |
|---|---|---:|
| `core_order_items` | `price <= 0` | 0 |
| `core_order_items` | `freight_value < 0` | 0 |
| `core_products` | `product_photos_qty < 0` | 0 |

This means that item prices, freight values, and product photo quantities do not show invalid values based on the defined rules.

---

### Data Quality Flags Created

Instead of deleting or modifying records immediately, numeric data quality flags were created to preserve the original data and mark invalid records.

| Table | Flag | Flagged Rows | Flagged % |
|---|---|---:|---:|
| `core_order_items` | `dq_price_non_positive` | 0 | 0.0000% |
| `core_order_items` | `dq_freight_negative` | 0 | 0.0000% |
| `core_payments` | `dq_payment_value_non_positive` | 9 | 0.0087% |
| `core_payments` | `dq_payment_installments_non_positive` | 2 | 0.0019% |
| `core_products` | `dq_product_weight_non_positive` | 6 | 0.0182% |
| `core_products` | `dq_product_length_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_height_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_width_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_photos_negative` | 0 | 0.0000% |

---

### Interpretation

The number of invalid numeric values is very small compared to the total size of the affected tables.

The most notable findings are:

- **9 payment records** have `payment_value <= 0`.
- **2 payment records** have `payment_installments <= 0`.
- **6 product records** have non-positive weight.
- **2 product records** have non-positive length.
- **2 product records** have non-positive height.
- **2 product records** have non-positive width.

These values are likely data quality exceptions or placeholder values. However, because the affected percentages are very small, these issues are not expected to significantly affect the overall dataset.

---

### Treatment Decision

No records were deleted at this stage.

The original numeric columns were kept unchanged, and data quality flags were created to identify invalid numeric records.

These flags will be used later during EDA and modeling to decide whether flagged records should be:

- excluded from specific calculations,
- analyzed separately,
- replaced with missing values,
- or handled as outliers/data quality exceptions.

---

### Final Decision

The numeric value validation issue is considered handled for the data quality preparation phase.

Further treatment, if needed, will be applied during EDA depending on the specific analysis objective.


# Problem No.7 
## Numeric Value Validation Summary

Numeric value validation was performed to identify values that are technically numeric but do not make business sense.

The validation focused on prices, payment values, installments, product weight, product dimensions, and product photo quantities.

---

### Validation Rules Applied

The following rules were checked:

#### Order Items
- `price <= 0`
- `freight_value < 0`

#### Payments
- `payment_value <= 0`
- `payment_installments <= 0`

#### Products
- `product_weight_g <= 0`
- `product_length_cm <= 0`
- `product_height_cm <= 0`
- `product_width_cm <= 0`
- `product_photos_qty < 0`

---

### Issues Found

A total of **6 numeric validation issues** were found.

| Table | Column | Issue | Total Rows | Invalid Rows | Invalid % |
|---|---|---|---:|---:|---:|
| `core_payments` | `payment_value` | `payment_value <= 0` | 103,886 | 9 | 0.0087% |
| `core_payments` | `payment_installments` | `payment_installments <= 0` | 103,886 | 2 | 0.0019% |
| `core_products` | `product_weight_g` | `product_weight_g <= 0` | 32,951 | 6 | 0.0182% |
| `core_products` | `product_length_cm` | `product_length_cm <= 0` | 32,951 | 2 | 0.0061% |
| `core_products` | `product_height_cm` | `product_height_cm <= 0` | 32,951 | 2 | 0.0061% |
| `core_products` | `product_width_cm` | `product_width_cm <= 0` | 32,951 | 2 | 0.0061% |

---

### Checks With No Issues

The following numeric checks returned **0 invalid rows**:

| Table | Check | Invalid Rows |
|---|---|---:|
| `core_order_items` | `price <= 0` | 0 |
| `core_order_items` | `freight_value < 0` | 0 |
| `core_products` | `product_photos_qty < 0` | 0 |

This means that item prices, freight values, and product photo quantities do not show invalid values based on the defined rules.

---

### Data Quality Flags Created

Instead of deleting or modifying records immediately, numeric data quality flags were created to preserve the original data and mark invalid records.

| Table | Flag | Flagged Rows | Flagged % |
|---|---|---:|---:|
| `core_order_items` | `dq_price_non_positive` | 0 | 0.0000% |
| `core_order_items` | `dq_freight_negative` | 0 | 0.0000% |
| `core_payments` | `dq_payment_value_non_positive` | 9 | 0.0087% |
| `core_payments` | `dq_payment_installments_non_positive` | 2 | 0.0019% |
| `core_products` | `dq_product_weight_non_positive` | 6 | 0.0182% |
| `core_products` | `dq_product_length_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_height_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_width_non_positive` | 2 | 0.0061% |
| `core_products` | `dq_product_photos_negative` | 0 | 0.0000% |

---

### Interpretation

The number of invalid numeric values is very small compared to the total size of the affected tables.

The most notable findings are:

- **9 payment records** have `payment_value <= 0`.
- **2 payment records** have `payment_installments <= 0`.
- **6 product records** have non-positive weight.
- **2 product records** have non-positive length.
- **2 product records** have non-positive height.
- **2 product records** have non-positive width.

These values are likely data quality exceptions or placeholder values. However, because the affected percentages are very small, these issues are not expected to significantly affect the overall dataset.

---

### Treatment Decision

No records were deleted at this stage.

The original numeric columns were kept unchanged, and data quality flags were created to identify invalid numeric records.

These flags will be used later during EDA and modeling to decide whether flagged records should be:

- excluded from specific calculations,
- analyzed separately,
- replaced with missing values,
- or handled as outliers/data quality exceptions.

---

### Final Decision

The numeric value validation issue is considered handled for the data quality preparation phase.

Further treatment, if needed, will be applied during EDA depending on the specific analysis objective.


# Problem No. 7
## Date Business Rule Validation

Date business rule validation checks whether timestamp columns follow a logical event sequence.

A date column may already be correctly converted to datetime, but the order of events may still be invalid.

### Rules Checked

#### Orders
- `order_purchase_timestamp <= order_approved_at`
- `order_approved_at <= order_delivered_carrier_date`
- `order_delivered_carrier_date <= order_delivered_customer_date`
- `order_purchase_timestamp <= order_delivered_customer_date`
- `order_purchase_timestamp <= order_estimated_delivery_date`

#### Reviews
- `review_creation_date <= review_answer_timestamp`

### Treatment Decision

Invalid date sequences will not be fixed or deleted immediately.

Instead, data quality flags will be created to identify records where the event sequence is not logically valid.

These flags will be used later during EDA and KPI calculations.

In [83]:
# Define date business validation rules.
# Each rule checks whether an earlier event date is less than or equal to a later event date.

date_business_rules = [
    {
        "table": "core_orders",
        "df": core_orders,
        "start_col": "order_purchase_timestamp",
        "end_col": "order_approved_at",
        "issue": "purchase timestamp after approval timestamp"
    },
    {
        "table": "core_orders",
        "df": core_orders,
        "start_col": "order_approved_at",
        "end_col": "order_delivered_carrier_date",
        "issue": "approval timestamp after carrier delivery timestamp"
    },
    {
        "table": "core_orders",
        "df": core_orders,
        "start_col": "order_delivered_carrier_date",
        "end_col": "order_delivered_customer_date",
        "issue": "carrier delivery timestamp after customer delivery timestamp"
    },
    {
        "table": "core_orders",
        "df": core_orders,
        "start_col": "order_purchase_timestamp",
        "end_col": "order_delivered_customer_date",
        "issue": "purchase timestamp after customer delivery timestamp"
    },
    {
        "table": "core_orders",
        "df": core_orders,
        "start_col": "order_purchase_timestamp",
        "end_col": "order_estimated_delivery_date",
        "issue": "purchase timestamp after estimated delivery date"
    },
    {
        "table": "core_reviews",
        "df": core_reviews,
        "start_col": "review_creation_date",
        "end_col": "review_answer_timestamp",
        "issue": "review creation date after review answer timestamp"
    }
]

In [84]:
# Run date business rule validation.
# A violation is counted only when both dates exist and the start date is after the end date.

date_rule_summary = []
date_invalid_rows = {}

for rule in date_business_rules:
    table_name = rule["table"]
    df = rule["df"]
    start_col = rule["start_col"]
    end_col = rule["end_col"]
    issue = rule["issue"]
    
    # Check only rows where both dates are available
    valid_date_pair_mask = df[start_col].notna() & df[end_col].notna()
    
    # Violation: start date is after end date
    invalid_mask = valid_date_pair_mask & (df[start_col] > df[end_col])
    
    invalid_rows = df[invalid_mask].copy()
    
    key = f"{table_name}: {start_col} > {end_col}"
    date_invalid_rows[key] = invalid_rows
    
    date_rule_summary.append({
        "table": table_name,
        "start_column": start_col,
        "end_column": end_col,
        "issue": issue,
        "rows_with_both_dates": valid_date_pair_mask.sum(),
        "invalid_rows": len(invalid_rows),
        "invalid_percentage_of_valid_pairs": round(
            len(invalid_rows) / valid_date_pair_mask.sum() * 100, 4
        ) if valid_date_pair_mask.sum() > 0 else 0
    })

date_rule_summary_df = pd.DataFrame(date_rule_summary)

print("✅ Date business rule validation completed")
display(date_rule_summary_df)

✅ Date business rule validation completed


,table,start_column,end_column,issue,rows_with_both_dates,invalid_rows,invalid_percentage_of_valid_pairs
0,core_orders,order_purchase_timestamp,order_approved_at,purchase timestamp after approval timestamp,99281,0,0.0000
1,core_orders,order_approved_at,order_delivered_carrier_date,approval timestamp after carrier delivery time...,97644,680,0.6964
2,core_orders,order_delivered_carrier_date,order_delivered_customer_date,carrier delivery timestamp after customer deli...,96475,20,0.0207
3,core_orders,order_purchase_timestamp,order_delivered_customer_date,purchase timestamp after customer delivery tim...,96476,0,0.0000
4,core_orders,order_purchase_timestamp,order_estimated_delivery_date,purchase timestamp after estimated delivery date,99441,0,0.0000
5,core_reviews,review_creation_date,review_answer_timestamp,review creation date after review answer times...,99222,18689,18.8355


In [85]:
# Show only date business rules that found invalid rows.

date_issues_found = date_rule_summary_df[
    date_rule_summary_df["invalid_rows"] > 0
].copy()

print("Number of date business rule issues found:", len(date_issues_found))

if len(date_issues_found) > 0:
    display(date_issues_found)
else:
    print("✅ No invalid date sequences found based on the defined rules.")

Number of date business rule issues found: 3


,table,start_column,end_column,issue,rows_with_both_dates,invalid_rows,invalid_percentage_of_valid_pairs
1,core_orders,order_approved_at,order_delivered_carrier_date,approval timestamp after carrier delivery time...,97644,680,0.6964
2,core_orders,order_delivered_carrier_date,order_delivered_customer_date,carrier delivery timestamp after customer deli...,96475,20,0.0207
5,core_reviews,review_creation_date,review_answer_timestamp,review creation date after review answer times...,99222,18689,18.8355


In [86]:
# Inspect invalid rows for each date business rule issue.
# This is for diagnosis only. No records are modified or deleted here.

for issue_name, invalid_df in date_invalid_rows.items():
    if len(invalid_df) > 0:
        print(f"\n===== {issue_name} =====")
        print("Invalid rows:", len(invalid_df))
        display(invalid_df.head(20))


===== core_orders: order_approved_at > order_delivered_carrier_date =====
Invalid rows: 680


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dq_delivered_missing_approved_at,dq_delivered_missing_carrier_date,dq_delivered_missing_customer_date,dq_delivered_any_missing_critical_date
51,002175704e8b209f61b9ad5cfd92b60e,a562db3c7cb9a68947debd30879b491e,delivered,2018-04-22,2018-04-24,2018-04-23,2018-05-02,2018-05-14,False,False,False,False
62,002834535f7a609a5c68266f173fa59e,89a6b1bfbe1b366503995b8df4c49450,delivered,2018-07-23,2018-07-28,2018-07-24,2018-08-11,2018-08-14,False,False,False,False
474,0133a47b53fee66218790a8d7dcab993,0663e652f8793c2d4eb77e8185a5747f,delivered,2018-04-20,2018-04-24,2018-04-23,2018-05-02,2018-05-22,False,False,False,False
580,0184d4ddb259e1a4cfc2871888cf97b8,09425ea1839abf2f0d289a0ff453fa21,delivered,2017-09-01,2017-09-13,2017-09-04,2017-09-09,2017-09-20,False,False,False,False
1236,0337e9398b087d93f41230ceb2410b50,deeec2dbc5ec2ebede3686654f7e82f0,delivered,2018-04-21,2018-04-24,2018-04-23,2018-04-27,2018-05-24,False,False,False,False
1329,036c6f5c3d6a81192f4896ddb2c00c83,7568855ab7b8c08837d50e8d9827090f,delivered,2018-07-22,2018-07-27,2018-07-24,2018-08-03,2018-08-16,False,False,False,False
1468,03c8468d3001db38cadd59ac670341cc,ecb809e65ff4dda823037481d5a64741,delivered,2018-07-22,2018-07-27,2018-07-24,2018-08-02,2018-08-07,False,False,False,False
1645,04323742e042716e4427a5b3536ea08f,ec2d2a258b058be97d409669ab436028,delivered,2018-04-23,2018-04-24,2018-04-23,2018-04-24,2018-05-10,False,False,False,False
1712,045eff272f42f0ae0191149686f193e1,9e2ae0d82b9057c99ad147658cd4e336,delivered,2018-04-23,2018-04-24,2018-04-23,2018-04-26,2018-05-10,False,False,False,False
1725,0467205a89711e4ec8e70ef2277e3287,90b8af517fbab96fb08d0115dffdc570,delivered,2018-07-03,2018-07-05,2018-07-03,2018-07-04,2018-07-16,False,False,False,False



===== core_orders: order_delivered_carrier_date > order_delivered_customer_date =====
Invalid rows: 20


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dq_delivered_missing_approved_at,dq_delivered_missing_carrier_date,dq_delivered_missing_customer_date,dq_delivered_any_missing_critical_date
3606,0922ee1619de7b995648e5a8407afb91,8dab637615e9eae6d33ef5e48644d6d3,delivered,2017-07-11,2017-07-11,2017-07-14,2017-07-12,2017-08-14,False,False,False,False
10094,19feb5627c41ea1b36a8e50a469b3644,b8097c8f0c1f58ab56a53812a446a898,delivered,2016-10-07,2016-10-07,2016-10-26,2016-10-20,2016-12-01,False,False,False,False
16049,29941903985f944b0ffc49c479c1547d,b56ee98181afc3948a758d73a08423de,delivered,2017-05-29,2017-05-29,2017-06-09,2017-06-02,2017-06-23,False,False,False,False
21785,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02,2017-07-02,2017-07-07,2017-07-06,2017-07-21,False,False,False,False
45898,76458889992169d3135b264dc13aec67,999196dca58a3db3d966d8f148532010,delivered,2016-10-07,2016-10-07,2016-10-26,2016-10-20,2016-11-29,False,False,False,False
46186,771c4f1f521f462e4b95619e648aaeab,d424b3ce4c850247e8c84ff8752de868,delivered,2017-03-22,2017-03-22,2017-03-30,2017-03-28,2017-04-12,False,False,False,False
54344,8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08,2016-10-08,2016-10-26,2016-10-25,2016-11-30,False,False,False,False
59320,99ad48402644a3968f2481defdc57947,abc709e0756f508111c68314b00cec25,delivered,2017-08-07,2017-08-07,2017-08-12,2017-08-10,2017-09-05,False,False,False,False
62499,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20,2017-07-21,2017-07-28,2017-07-25,2017-08-14,False,False,False,False
69189,b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14,2017-06-14,2017-06-27,2017-06-26,2017-07-14,False,False,False,False



===== core_reviews: review_creation_date > review_answer_timestamp =====
Invalid rows: 18689


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_row_id,has_review_comment_title,has_review_comment_message,has_any_review_text
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-10-03,2018-03-11 03:05:00,2,False,False,False
22,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10,2018-10-07,2018-07-11 14:10:00,23,True,True,True
32,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,1,NaN,Sempre compro pela Internet e a entrega ocorre...,2018-08-04,2018-04-09 12:22:00,33,False,True,True
34,c92cdd7dd544a01aa35137f901669cdf,37e7875cdce5a9e5b3a692971f370151,4,Muito bom.,Recebi exatamente o que esperava. As demais en...,2018-07-06,2018-06-09 18:44:00,35,True,True,True
35,c0f4c8301f8c29d6f53d239e194bb4c2,04db86e3493b3724ac4bd3c9a397e23c,5,NaN,NaN,2017-12-09,2017-09-13 14:33:00,36,False,False,False
38,b193ff3c9f32a01f3a0d9ae26b94d244,e2e6ee1ed2d7f2f36b05d234983bd7a0,5,Maravilhoso!,"Tô completamente apaixonada, loja super respon...",2018-10-08,2018-08-11 00:22:00,39,True,True,True
41,eb26c2bfb5030f57dcef30d3f111eb1e,88096595631e8d26b658306256e3ce34,1,NaN,NaN,2018-08-02,2018-02-10 04:28:00,42,False,False,False
44,6f3eb43ec2f5092cf04c8d705864a8ac,f3a9c397a2fa5a2366b4b96bc15737f6,5,NaN,NaN,2018-11-03,2018-03-14 00:38:00,45,False,False,False
56,cb6fe4f91f83d9222b7c8e14959055e1,84226f5c73669d7711ca44aa9cac602a,3,NaN,NaN,2018-10-05,2018-05-13 22:51:00,57,False,False,False
69,750d4311e8ddb6cba549ee86f06f0cca,4b32a4cdfc1b80066a8ee3a4b7a2094f,5,NaN,"Produto entregue como solicitado, e com muita ...",2017-09-08,2017-08-09 12:03:00,70,False,True,True


In [87]:
# Create date logic quality flags.
# These flags preserve original date values while marking invalid event sequences.

# core_orders date logic flags
core_orders["dq_purchase_after_approval"] = (
    core_orders["order_purchase_timestamp"].notna() &
    core_orders["order_approved_at"].notna() &
    (core_orders["order_purchase_timestamp"] > core_orders["order_approved_at"])
)

core_orders["dq_approval_after_carrier"] = (
    core_orders["order_approved_at"].notna() &
    core_orders["order_delivered_carrier_date"].notna() &
    (core_orders["order_approved_at"] > core_orders["order_delivered_carrier_date"])
)

core_orders["dq_carrier_after_customer"] = (
    core_orders["order_delivered_carrier_date"].notna() &
    core_orders["order_delivered_customer_date"].notna() &
    (core_orders["order_delivered_carrier_date"] > core_orders["order_delivered_customer_date"])
)

core_orders["dq_purchase_after_customer_delivery"] = (
    core_orders["order_purchase_timestamp"].notna() &
    core_orders["order_delivered_customer_date"].notna() &
    (core_orders["order_purchase_timestamp"] > core_orders["order_delivered_customer_date"])
)

core_orders["dq_purchase_after_estimated_delivery"] = (
    core_orders["order_purchase_timestamp"].notna() &
    core_orders["order_estimated_delivery_date"].notna() &
    (core_orders["order_purchase_timestamp"] > core_orders["order_estimated_delivery_date"])
)

# core_reviews date logic flag
core_reviews["dq_review_creation_after_answer"] = (
    core_reviews["review_creation_date"].notna() &
    core_reviews["review_answer_timestamp"].notna() &
    (core_reviews["review_creation_date"] > core_reviews["review_answer_timestamp"])
)

print("✅ Date logic quality flags created")

✅ Date logic quality flags created


In [88]:
# Summarize date logic quality flags across affected tables.

date_flag_summary = []

date_flag_configs = [
    ("core_orders", core_orders, [
        "dq_purchase_after_approval",
        "dq_approval_after_carrier",
        "dq_carrier_after_customer",
        "dq_purchase_after_customer_delivery",
        "dq_purchase_after_estimated_delivery"
    ]),
    ("core_reviews", core_reviews, [
        "dq_review_creation_after_answer"
    ])
]

for table_name, df, flags in date_flag_configs:
    for flag in flags:
        flagged_rows = df[flag].sum()
        date_flag_summary.append({
            "table": table_name,
            "flag": flag,
            "flagged_rows": flagged_rows,
            "flagged_percentage": round(flagged_rows / len(df) * 100, 4)
        })

date_flag_summary_df = pd.DataFrame(date_flag_summary)

display(date_flag_summary_df)

,table,flag,flagged_rows,flagged_percentage
0,core_orders,dq_purchase_after_approval,0,0.0000
1,core_orders,dq_approval_after_carrier,680,0.6838
2,core_orders,dq_carrier_after_customer,20,0.0201
3,core_orders,dq_purchase_after_customer_delivery,0,0.0000
4,core_orders,dq_purchase_after_estimated_delivery,0,0.0000
5,core_reviews,dq_review_creation_after_answer,18689,18.8355


## Date Business Rule Validation Summary

Date business_at`Date business rule validation was performed to verify that timestamp columns follow a logical event sequence.
- `order_approved_at <= order_delivered_carrier_date`
- `order_delivered_carrier_date <= order_delivered_customer_date`
- `order_purchase_timestamp <= order_delivered_customer_date`
- `order_purchase_timestamp <= order_estimated_delivery_date`

#### Reviews
- `review_creation_date <= review_answer_timestamp`

Rows with missing values in either date column were excluded from each rule check.  
Only rows where both dates were available were evaluated.

---

### Issues Found

A total of **3 date business rule issues** were found.

| Table | Rule Violated | Rows With Both Dates | Invalid Rows | Invalid % of Valid Pairs |
|---|---|---:|---:|---:|
| `core_orders` | `order_approved_at > order_delivered_carrier_date` | 97,644 | 680 | 0.6964% |
| `core_orders` | `order_delivered_carrier_date > order_delivered_customer_date` | 96,475 | 20 | 0.0207% |
| `core_reviews` | `review_creation_date > review_answer_timestamp` | 99,222 | 18,689 | 18.8355% |

---

### Checks With No Issues

The following rules returned **0 invalid rows**:

| Table | Rule Checked | Invalid Rows |
|---|---|---:|
| `core_orders` | `order_purchase_timestamp > order_approved_at` | 0 |
| `core_orders` | `order_purchase_timestamp > order_delivered_customer_date` | 0 |
| `core_orders` | `order_purchase_timestamp > order_estimated_delivery_date` | 0 |

This means that the purchase timestamp is logically consistent with approval, customer delivery, and estimated delivery dates based on the defined rules.

---

### Data Quality Flags Created

Instead of modifying or deleting records, date logic data quality flags were created to preserve the original data and mark invalid event sequences.

| Table | Flag | Flagged Rows | Flagged % |
|---|---|---:|---:|
| `core_orders` | `dq_purchase_after_approval` | 0 | 0.0000% |
| `core_orders` | `dq_approval_after_carrier` | 680 | 0.6838% |
| `core_orders` | `dq_carrier_after_customer` | 20 | 0.0201% |
| `core_orders` | `dq_purchase_after_customer_delivery` | 0 | 0.0000% |
| `core_orders` | `dq_purchase_after_estimated_delivery` | 0 | 0.0000% |
| `core_reviews` | `dq_review_creation_after_answer` | 18,689 | 18.8355% |

---

### Interpretation

The order-related date issues are relatively small compared to the full dataset:

- **680 orders** have approval timestamps after carrier delivery timestamps.
- **20 orders** have carrier delivery timestamps after customer delivery timestamps.

These are likely timestamp recording inconsistencies or operational logging issues.  
Because the percentages are small, these issues are not expected to significantly affect the full dataset, but they should be excluded or flagged in delivery-time calculations.

The review-related issue is much larger:

- **18,689 reviews** have `review_creation_date` after `review_answer_timestamp`.

This suggests that the meaning or granularity of these review timestamps should be handled carefully.  
For example, `review_creation_date` may be stored as a date-level value while `review_answer_timestamp` may be stored as a timestamp-level value, or there may be system-generated timestamp inconsistencies.

---

### Treatment Decision

No original date columns were modified.

Instead:

- Invalid date sequences were flagged.
- Records were not deleted.
- The created flags will be used later during EDA and KPI calculations.
- Delivery KPI calculations should exclude or separately analyze records flagged by:
  - `dq_approval_after_carrier`
  - `dq_carrier_after_customer`
- Review timing analysis should account for:
  - `dq_review_creation_after_answer`

---

### Final Decision

The date business rule validation issue is considered handled for the data quality preparation phase.

The original timestamps remain unchanged, and all invalid date sequences are now identifiable using data quality flags.

Further treatment will be decided during EDA depending on the analysis objective.

This step is different from datetime conversion.  
Datetime conversion confirms that values can be read as dates, while date business validation checks whether the sequence of events makes business sense.

---

### Rules Checked

The following date sequence rules were validated:

#### Orders


# Problem No. 8
## Categorical Value Consistency

Categorical value consistency checks are performed to ensure that categorical columns contain expected, clean, and standardized values.

Categorical columns often have repeated values by nature, so repetition is not a problem. The goal is to identify unexpected categories, spelling inconsistencies, blank values, rare categories, or formatting issues such as leading/trailing spaces.

### Columns Checked

The following categorical columns will be reviewed:

- `core_orders.order_status`
- `core_payments.payment_type`
- `core_customers.customer_state`
- `core_sellers.seller_state`
- `ref_geolocation_zip.geolocation_state`
- `core_products.product_category_name`
- `ref_category_translation.product_category_name`
- `ref_category_translation.product_category_name_english`

### Treatment Decision

At this stage, categorical values will not be changed automatically.

We will first:
- list unique values,
- count their frequencies,
- detect blank or whitespace-only values,
- compare state codes across related tables,
- check whether product categories match the translation table.

Any issue found will be reviewed before applying treatment.

In [89]:
# Define categorical columns to validate.
# These columns are expected to contain a limited set of repeated categories.

categorical_checks = [
    {
        "table": "core_orders",
        "df": core_orders,
        "column": "order_status"
    },
    {
        "table": "core_payments",
        "df": core_payments,
        "column": "payment_type"
    },
    {
        "table": "core_customers",
        "df": core_customers,
        "column": "customer_state"
    },
    {
        "table": "core_sellers",
        "df": core_sellers,
        "column": "seller_state"
    },
    {
        "table": "ref_geolocation_zip",
        "df": ref_geolocation_zip,
        "column": "geolocation_state"
    },
    {
        "table": "core_products",
        "df": core_products,
        "column": "product_category_name"
    },
    {
        "table": "ref_category_translation",
        "df": ref_category_translation,
        "column": "product_category_name"
    },
    {
        "table": "ref_category_translation",
        "df": ref_category_translation,
        "column": "product_category_name_english"
    }
]

In [90]:
# Profile categorical columns for missing values, blank strings, unique values, and spacing issues.

categorical_profile = []

for check in categorical_checks:
    table_name = check["table"]
    df = check["df"]
    column = check["column"]
    
    series = df[column]
    cleaned_series = series.fillna("").astype(str)
    stripped_series = cleaned_series.str.strip()
    
    missing_count = series.isna().sum()
    blank_count = stripped_series.eq("").sum()
    unique_count = stripped_series[~stripped_series.eq("")].nunique()
    
    leading_trailing_space_count = (
        cleaned_series.ne(stripped_series) & ~stripped_series.eq("")
    ).sum()
    
    top_value = stripped_series[~stripped_series.eq("")].mode()
    top_value = top_value.iloc[0] if len(top_value) > 0 else None
    
    top_value_count = (
        stripped_series.eq(top_value).sum() if top_value is not None else 0
    )
    
    categorical_profile.append({
        "table": table_name,
        "column": column,
        "total_rows": len(df),
        "missing_count": missing_count,
        "blank_or_whitespace_count": blank_count,
        "unique_values": unique_count,
        "leading_trailing_space_count": leading_trailing_space_count,
        "most_frequent_value": top_value,
        "most_frequent_value_count": top_value_count,
        "most_frequent_value_pct": round(top_value_count / len(df) * 100, 2)
    })

categorical_profile_df = pd.DataFrame(categorical_profile)

print("✅ Categorical profiling completed")
display(categorical_profile_df)

✅ Categorical profiling completed


,table,column,total_rows,missing_count,blank_or_whitespace_count,unique_values,leading_trailing_space_count,most_frequent_value,most_frequent_value_count,most_frequent_value_pct
0,core_orders,order_status,99441,0,0,8,0,delivered,96478,97.02
1,core_payments,payment_type,103886,0,0,5,0,credit_card,76795,73.92
2,core_customers,customer_state,99441,0,0,27,0,SP,41746,41.98
3,core_sellers,seller_state,3095,0,0,23,0,SP,1849,59.74
4,ref_geolocation_zip,geolocation_state,19015,0,0,27,0,SP,6349,33.39
5,core_products,product_category_name,32951,0,0,74,0,cama_mesa_banho,3029,9.19
6,ref_category_translation,product_category_name,74,0,0,74,0,agro_industria_e_comercio,1,1.35
7,ref_category_translation,product_category_name_english,74,0,0,74,0,PC gamer,1,1.35


In [91]:
# Display value counts for each categorical column.
# This helps identify unexpected or rare categories.

for check in categorical_checks:
    table_name = check["table"]
    df = check["df"]
    column = check["column"]
    
    print(f"\n===== {table_name}.{column} =====")
    
    value_counts_df = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .reset_index()
    )
    
    value_counts_df.columns = [column, "count"]
    value_counts_df["percentage"] = (
        value_counts_df["count"] / len(df) * 100
    ).round(2)
    
    display(value_counts_df.head(30))


===== core_orders.order_status =====


,order_status,count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00



===== core_payments.payment_type =====


,payment_type,count,percentage
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00



===== core_customers.customer_state =====


...

,product_category_name,count,percentage
0,cama_mesa_banho,3029,9.19
1,esporte_lazer,2867,8.70
2,moveis_decoracao,2657,8.06
3,beleza_saude,2444,7.42
4,utilidades_domesticas,2335,7.09
5,automotivo,1900,5.77
6,informatica_acessorios,1639,4.97
7,brinquedos,1411,4.28
8,relogios_presentes,1329,4.03
9,telefonia,1134,3.44



===== ref_category_translation.product_category_name =====


,product_category_name,count,percentage
0,agro_industria_e_comercio,1,1.35
1,alimentos,1,1.35
2,alimentos_bebidas,1,1.35
3,artes,1,1.35
4,artes_e_artesanato,1,1.35
5,artigos_de_festas,1,1.35
6,artigos_de_natal,1,1.35
7,audio,1,1.35
8,automotivo,1,1.35
9,bebes,1,1.35



===== ref_category_translation.product_category_name_english =====


,product_category_name_english,count,percentage
0,agro_industry_and_commerce,1,1.35
1,food,1,1.35
2,food_drink,1,1.35
3,art,1,1.35
4,arts_and_craftmanship,1,1.35
5,party_supplies,1,1.35
6,christmas_supplies,1,1.35
7,audio,1,1.35
8,auto,1,1.35
9,baby,1,1.35


In [92]:
# Define expected values for key categorical columns.
# Values outside these sets will be flagged as unexpected.

expected_values = {
    "core_orders.order_status": {
        "delivered", "shipped", "canceled", "unavailable",
        "invoiced", "processing", "created", "approved"
    },
    "core_payments.payment_type": {
        "credit_card", "boleto", "voucher", "debit_card", "not_defined"
    }
}

unexpected_category_results = []

for check in categorical_checks:
    table_name = check["table"]
    df = check["df"]
    column = check["column"]
    full_name = f"{table_name}.{column}"
    
    if full_name in expected_values:
        observed_values = set(
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )
        
        unexpected_values = sorted(observed_values - expected_values[full_name])
        missing_expected_values = sorted(expected_values[full_name] - observed_values)
        
        unexpected_category_results.append({
            "column": full_name,
            "unexpected_values_count": len(unexpected_values),
            "unexpected_values": unexpected_values,
            "missing_expected_values_count": len(missing_expected_values),
            "missing_expected_values": missing_expected_values
        })

unexpected_category_results_df = pd.DataFrame(unexpected_category_results)

print("✅ Expected value checks completed")
display(unexpected_category_results_df)

✅ Expected value checks completed


,column,unexpected_values_count,unexpected_values,missing_expected_values_count,missing_expected_values
0,core_orders.order_status,0,[],0,[]
1,core_payments.payment_type,0,[],0,[]


In [93]:
# Compare state codes across customers, sellers, and geolocation.
# This helps identify inconsistent or unmatched state codes.

customer_states = set(core_customers["customer_state"].dropna().astype(str).str.strip().unique())
seller_states = set(core_sellers["seller_state"].dropna().astype(str).str.strip().unique())
geo_states = set(ref_geolocation_zip["geolocation_state"].dropna().astype(str).str.strip().unique())

state_consistency_summary = pd.DataFrame({
    "check": [
        "Customer states not in geolocation states",
        "Seller states not in geolocation states",
        "Geolocation states not in customer states",
        "Geolocation states not in seller states"
    ],
    "count": [
        len(customer_states - geo_states),
        len(seller_states - geo_states),
        len(geo_states - customer_states),
        len(geo_states - seller_states)
    ],
    "values": [
        sorted(customer_states - geo_states),
        sorted(seller_states - geo_states),
        sorted(geo_states - customer_states),
        sorted(geo_states - seller_states)
    ]
})

print("✅ State consistency check completed")
display(state_consistency_summary)

✅ State consistency check completed


,check,count,values
0,Customer states not in geolocation states,0,[]
1,Seller states not in geolocation states,0,[]
2,Geolocation states not in customer states,0,[]
3,Geolocation states not in seller states,4,"[AL, AP, RR, TO]"


In [94]:
# Check whether product categories in core_products exist in ref_category_translation.

product_categories = set(
    core_products["product_category_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

translation_categories = set(
    ref_category_translation["product_category_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

product_categories_missing_translation = sorted(product_categories - translation_categories)
translation_categories_not_used = sorted(translation_categories - product_categories)

category_consistency_summary = pd.DataFrame({
    "check": [
        "Product categories missing from translation table",
        "Translation categories not used in products"
    ],
    "count": [
        len(product_categories_missing_translation),
        len(translation_categories_not_used)
    ],
    "values": [
        product_categories_missing_translation,
        translation_categories_not_used
    ]
})

print("✅ Product category consistency check completed")
display(category_consistency_summary)

✅ Product category consistency check completed


,check,count,values
0,Product categories missing from translation table,0,[]
1,Translation categories not used in products,0,[]


In [95]:
# Create a summary of categorical columns with leading/trailing spaces.
# This helps decide whether text standardization is needed.

categorical_spacing_issues = categorical_profile_df[
    categorical_profile_df["leading_trailing_space_count"] > 0
].copy()

print("Number of categorical columns with leading/trailing spaces:", len(categorical_spacing_issues))

if len(categorical_spacing_issues) > 0:
    display(categorical_spacing_issues)
else:
    print("✅ No leading/trailing space issues found in categorical columns.")

Number of categorical columns with leading/trailing spaces: 0
✅ No leading/trailing space issues found in categorical columns.


## Categorical Value Consistency Summary

Categorical trailing space issues were detected.Categorical value consistency checks were performed to ensure that key categorical columns contain clean, expected, and standardized values.

| Table | Column | Total Rows | Missing Count | Blank/Whitespace Count | Unique Values | Leading/Trailing Spaces | Most Frequent Value | Most Frequent % |
|---|---|---:|---:|---:|---:|---:|---|---:|
| `core_orders` | `order_status` | 99,441 | 0 | 0 | 8 | 0 | `delivered` | 97.02% |
| `core_payments` | `payment_type` | 103,886 | 0 | 0 | 5 | 0 | `credit_card` | 73.92% |
| `core_customers` | `customer_state` | 99,441 | 0 | 0 | 27 | 0 | `SP` | 41.98% |
| `core_sellers` | `seller_state` | 3,095 | 0 | 0 | 23 | 0 | `SP` | 59.74% |
| `ref_geolocation_zip` | `geolocation_state` | 19,015 | 0 | 0 | 27 | 0 | `SP` | 33.39% |
| `core_products` | `product_category_name` | 32,951 | 0 | 0 | 74 | 0 | `cama_mesa_banho` | 9.19% |
| `ref_category_translation` | `product_category_name` | 74 | 0 | 0 | 74 | 0 | `agro_industria_e_comercio` | 1.35% |
| `ref_category_translation` | `product_category_name_english` | 74 | 0 | 0 | 74 | 0 | `PC gamer` | 1.35% |

---

### Expected Value Checks

Expected value checks were performed for key controlled categorical columns.

#### `core_orders.order_status`

All observed order statuses matched the expected set of values.

Observed values included:

- `delivered`
- `shipped`
- `canceled`
- `unavailable`
- `invoiced`
- `processing`
- `created`
- `approved`

Result:

- Unexpected values: **0**
- Missing expected values: **0**

#### `core_payments.payment_type`

All observed payment types matched the expected set of values.

Observed values included:

- `credit_card`
- `boleto`
- `voucher`
- `debit_card`
- `not_defined`

Result:

- Unexpected values: **0**
- Missing expected values: **0**

---

### Key Distribution Insights

#### Order Status

The majority of orders are marked as `delivered`.

| Order Status | Count | Percentage |
|---|---:|---:|
| `delivered` | 96,478 | 97.02% |
| `shipped` | 1,107 | 1.11% |
| `canceled` | 625 | 0.63% |
| `unavailable` | 609 | 0.61% |
| `invoiced` | 314 | 0.32% |
| `processing` | 301 | 0.30% |
| `created` | 5 | 0.01% |
| `approved` | 2 | 0.00% |

This distribution is expected for an orders dataset where most completed transactions are delivered.

#### Payment Type

Most payments were made by credit card.

| Payment Type | Count | Percentage |
|---|---:|---:|
| `credit_card` | 76,795 | 73.92% |
| `boleto` | 19,784 | 19.04% |
| `voucher` | 5,775 | 5.56% |
| `debit_card` | 1,529 | 1.47% |
| `not_defined` | 3 | 0.00% |

The `not_defined` category exists but is extremely rare, with only 3 records.  
This value is valid based on the expected payment type set and does not require treatment at this stage.

---

### State Code Consistency

State code consistency was checked across:

- `core_customers.customer_state`
- `core_sellers.seller_state`
- `ref_geolocation_zip.geolocation_state`

#### Results

| Check | Count | Values |
|---|---:|---|
| Customer states not in geolocation states | 0 | `[]` |
| Seller states not in geolocation states | 0 | `[]` |
| Geolocation states not in customer states | 0 | `[]` |
| Geolocation states not in seller states | 4 | `[AL, AP, RR, TO]` |

### Interpretation

All customer and seller states exist in the geolocation reference table.

The geolocation table contains 4 states that do not appear in the seller table:

- `AL`
- `AP`
- `RR`
- `TO`

This is not a data quality issue.  
It simply means there are geolocation records for these states, but no sellers are registered in these states in the current seller dataset.

No treatment is required.

---

### Product Category Consistency

Product categories were checked between:

- `core_products.product_category_name`
- `ref_category_translation.product_category_name`

#### Results

| Check | Count | Values |
|---|---:|---|
| Product categories missing from translation table | 0 | `[]` |
| Translation categories not used in products | 0 | `[]` |

### Interpretation

All product categories in `core_products` exist in the category translation table.

Also, all categories in the translation table are used in the products table.

This confirms that product category mapping is complete and consistent.

---

### Spacing and Formatting Issues

A spacing check was performed to detect leading or trailing spaces in categorical columns.

Result:

- Number of categorical columns with leading/trailing spaces: **0**

No text standardization was required for spacing issues.

---

### Treatment Decision

No categorical values were modified.

No records were removed.

No category mapping changes were required.

The categorical columns are considered clean and consistent based on the checks performed.

---

### Final Decision

The categorical value consistency issue is considered resolved.

The dataset has:

- no unexpected `order_status` values,
- no unexpected `payment_type` values,
- no missing or blank categorical values in reviewed columns,
- no leading/trailing space issues,
- consistent state codes across customers, sellers, and geolocation,
- and complete product category mapping with the translation table.

These categorical fields are ready for EDA and downstream analysis.

The goal of this check was to identify:
- unexpected categorical values,
- missing or blank category values,
- leading/trailing spaces,
- inconsistent state codes,
- unmatched product categories,
- and category translation inconsistencies.

---

### Columns Reviewed

The following categorical columns were checked:

- `core_orders.order_status`
- `core_payments.payment_type`
- `core_customers.customer_state`
- `core_sellers.seller_state`
- `ref_geolocation_zip.geolocation_state`
- `core_products.product_category_name`
- `ref_category_translation.product_category_name`
- `ref_category_translation.product_category_name_english`

---

### General Categorical Profiling Results

No missing, blank, or whitespace-only values were found in the reviewed categorical columns.



# Final Clean Dataset Export

After completing the main data quality preparation steps, the cleaned and enriched datasets will be exported as CSV files.

### Purpose

The goal of this step is to save the final working versions of the datasets after applying:

- CSV parsing fixes
- datetime conversion
- missing value treatment
- data quality flags
- duplicate handling
- category translation fixes
- geolocation ZIP-level aggregation
- referential integrity flags
- numeric validation flags
- date business rule flags

### Export Decision

The original raw files will remain unchanged.

Cleaned and analysis-ready files will be exported into a dedicated folder:

`clean_exports`

These exported files can be reused later in:

- EDA
- SQL
- Power BI
- machine learning
- reporting

In [96]:
from pathlib import Path

# Create a folder to store cleaned exported CSV files.
# If the folder already exists, it will not raise an error.

export_folder = Path("clean_exports")
export_folder.mkdir(exist_ok=True)

print("✅ Export folder is ready:", export_folder.resolve())

✅ Export folder is ready: C:\Users\safwatr\IntelGraphicsProfiles\clean_exports


In [97]:
# Define the final datasets to export.
# These are the cleaned/enriched versions currently available in memory.

datasets_to_export = {
    "core_customers_clean": core_customers,
    "core_orders_clean": core_orders,
    "core_order_items_clean": core_order_items,
    "core_payments_clean": core_payments,
    "core_products_clean": core_products,
    "core_reviews_clean": core_reviews,
    "core_sellers_clean": core_sellers,
    "ref_category_translation_clean": ref_category_translation,
    "ref_geolocation_clean": ref_geolocation_clean,
    "ref_geolocation_zip": ref_geolocation_zip
}

print("✅ Datasets prepared for export:")
for name in datasets_to_export.keys():
    print("-", name)

✅ Datasets prepared for export:
- core_customers_clean
- core_orders_clean
- core_order_items_clean
- core_payments_clean
- core_products_clean
- core_reviews_clean
- core_sellers_clean
- ref_category_translation_clean
- ref_geolocation_clean
- ref_geolocation_zip


In [98]:
# Export each cleaned dataset to CSV.
# utf-8-sig is used for better compatibility with Excel.

exported_files = []

for dataset_name, df in datasets_to_export.items():
    file_path = export_folder / f"{dataset_name}.csv"
    
    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )
    
    exported_files.append({
        "dataset": dataset_name,
        "rows": len(df),
        "columns": df.shape[1],
        "file_path": str(file_path)
    })

export_summary_df = pd.DataFrame(exported_files)

print("✅ All cleaned datasets exported successfully")
display(export_summary_df)

✅ All cleaned datasets exported successfully


,dataset,rows,columns,file_path
0,core_customers_clean,99441,6,clean_exports\core_customers_clean.csv
1,core_orders_clean,99441,17,clean_exports\core_orders_clean.csv
2,core_order_items_clean,112650,9,clean_exports\core_order_items_clean.csv
3,core_payments_clean,103886,7,clean_exports\core_payments_clean.csv
4,core_products_clean,32951,14,clean_exports\core_products_clean.csv
5,core_reviews_clean,99222,12,clean_exports\core_reviews_clean.csv
6,core_sellers_clean,3095,5,clean_exports\core_sellers_clean.csv
7,ref_category_translation_clean,74,2,clean_exports\ref_category_translation_clean.csv
8,ref_geolocation_clean,738332,5,clean_exports\ref_geolocation_clean.csv
9,ref_geolocation_zip,19015,5,clean_exports\ref_geolocation_zip.csv


In [99]:
# Verify that all exported CSV files exist in the export folder.

export_verification = []

for dataset_name in datasets_to_export.keys():
    file_path = export_folder / f"{dataset_name}.csv"
    
    export_verification.append({
        "dataset": dataset_name,
        "file_exists": file_path.exists(),
        "file_size_mb": round(file_path.stat().st_size / (1024 * 1024), 3) if file_path.exists() else None,
        "file_path": str(file_path)
    })

export_verification_df = pd.DataFrame(export_verification)

print("✅ Export verification completed")
display(export_verification_df)

✅ Export verification completed


,dataset,file_exists,file_size_mb,file_path
0,core_customers_clean,True,8.735,clean_exports\core_customers_clean.csv
1,core_orders_clean,True,17.591,clean_exports\core_orders_clean.csv
2,core_order_items_clean,True,16.692,clean_exports\core_order_items_clean.csv
3,core_payments_clean,True,6.660,clean_exports\core_payments_clean.csv
4,core_products_clean,True,3.210,clean_exports\core_products_clean.csv
5,core_reviews_clean,True,16.123,clean_exports\core_reviews_clean.csv
6,core_sellers_clean,True,0.174,clean_exports\core_sellers_clean.csv
7,ref_category_translation_clean,True,0.003,clean_exports\ref_category_translation_clean.csv
8,ref_geolocation_clean,True,42.030,clean_exports\ref_geolocation_clean.csv
9,ref_geolocation_zip,True,1.088,clean_exports\ref_geolocation_zip.csv


In [100]:
# Save the export summary as a CSV log file.

export_summary_path = export_folder / "export_summary.csv"

export_summary_df.to_csv(
    export_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Export summary saved:", export_summary_path)

✅ Export summary saved: clean_exports\export_summary.csv


## Final Clean Dataset Export Summary

All cleaned and analysis-ready datasets were exported successfully.

### Export Status

The final cleaned datasets were saved in the `clean_exports` folder.

### Exported Files

| Dataset | Rows | Columns | File Path |
|---|---:|---:|---|
| `core_customers_clean` | 99,441 | 6 | `clean_exports\core_customers_clean.csv` |
| `core_orders_clean` | 99,441 | 17 | `clean_exports\core_orders_clean.csv` |
| `core_order_items_clean` | 112,650 | 9 | `clean_exports\core_order_items_clean.csv` |
| `core_payments_clean` | 103,886 | 7 | `clean_exports\core_payments_clean.csv` |
| `core_products_clean` | 32,951 | 14 | `clean_exports\core_products_clean.csv` |
| `core_reviews_clean` | 99,222 | 12 | `clean_exports\core_reviews_clean.csv` |
| `core_sellers_clean` | 3,095 | 5 | `clean_exports\core_sellers_clean.csv` |
| `ref_category_translation_clean` | 74 | 2 | `clean_exports\ref_category_translation_clean.csv` |
| `ref_geolocation_clean` | 738,332 | 5 | `clean_exports\ref_geolocation_clean.csv` |
| `ref_geolocation_zip` | 19,015 | 5 | `clean_exports\ref_geolocation_zip.csv` |

### Exported Dataset Notes

- `ref_category_translation_clean` contains the cleaned category translation table.
- `ref_geolocation_clean` contains the geolocation table after removing exact full duplicate rows.
- `ref_geolocation_zip` contains the ZIP-level geolocation lookup table with one row per ZIP code.
- Core tables were exported after applying the relevant data quality preparation steps and flags.

### Decision

The cleaned datasets are now ready for:

- EDA
- SQL loading
- Power BI dashboards
- reporting
- modeling
- further analytical workflows

The raw source files remain unchanged, while the exported files represent the cleaned working versions of the datasets.